This notebook provides the core code accompanying the project titled "Addressing the Rashomon Effect in Machine Learning-Based Discovery of Water Splitting Dual-Atom Catalysts Through Cross-Model Consensus".

# 1. Import Libraries

In [ ]:
#Import libraries

#Warning suppression
import warnings
warnings.filterwarnings('ignore')

#General
import os
import re
import glob
import time
import joblib
import itertools
from itertools import combinations
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from matplotlib.patches import Patch, Ellipse, Circle
from scipy.stats import spearmanr, rankdata, randint, uniform, loguniform

#Descriptors
from mendeleev import element
from matminer.utils.data import MagpieData
from pymatgen.core import Element
from sklearn.feature_selection import mutual_info_regression
from boruta import BorutaPy
from sklearn.base import clone

#Machine learning
from sklearn.utils import check_random_state
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

#Baseline model
from sklearn.dummy import DummyRegressor

#Machine learning models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
from sklearn.neural_network import MLPRegressor
from pysr import PySRRegressor
import shap

# 2. Dataset Preparation

In [ ]:
#Define file paths
    #All inputs are read from, and all outputs written to, base_dir
base_dir = os.path.expanduser("~/Desktop") 
os.makedirs(base_dir, exist_ok=True)
data_folder = os.path.join(base_dir, "data", "all_data")
slab_file = os.path.join(base_dir, "data", "slab_data", "final_energies_slab.csv")
output_file = os.path.join(base_dir, "dataset.csv")

#Reference energies (Sun, 2026)
E_H2 = -6.8 #eV 
E_H2O = -14.22 #eV 

#Zero-point energy and entropy corrections (Norskov et al., 2005; Rossmeisl et al., 2007)
ZPE_H = 0.24 #eV — combined ΔZPE - TΔS for H
ZPE_OH = 0.35 #eV — combined ΔZPE - TΔS for OH
ZPE_O = 0.05 #eV — combined ΔZPE - TΔS for O
ZPE_OOH = 0.40 #eV — combined ΔZPE - TΔS for OOH

#Locate and load slab and adsorbate energy files
file_pattern = os.path.join(data_folder, "final_energies_*.csv")
files = glob.glob(file_pattern)
print(f"Found {len(files)} slab and adsorbate energy files: {[os.path.basename(f) for f in files]}")

#Process slab and adsorbate energy files
all_dfs = []
for filepath in files:
    filename = os.path.basename(filepath)
    intermediate = re.sub(r'^final_energies_', '', os.path.splitext(filename)[0])
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        df.columns[0]: 'metal_A',
        df.columns[1]: 'metal_B',
        df.columns[2]: 'energy_slab_adsorbate'
    })
    df['metal_A'] = df['metal_A'].str.strip()
    df['metal_B'] = df['metal_B'].str.strip()
    df['metal_group'] = df['metal_A'] + df['metal_B']
    df['intermediate'] = intermediate
    all_dfs.append(df)

#Combine slab and adsorbate energy files
dataset_long = pd.concat(all_dfs, ignore_index=True)
target_intermediates = ['H', 'O', 'OH', 'OOH']
dataset_long = dataset_long[dataset_long['intermediate'].isin(target_intermediates)].copy()
dataset_long = dataset_long.reset_index(drop=True)

#Load slab energy file
df_slab = pd.read_csv(slab_file)
df_slab = df_slab.rename(columns={
    df_slab.columns[0]: 'metal_A',
    df_slab.columns[1]: 'metal_B',
    df_slab.columns[2]: 'energy_slab'
})
df_slab['metal_A'] = df_slab['metal_A'].str.strip()
df_slab['metal_B'] = df_slab['metal_B'].str.strip()
df_slab['metal_group'] = df_slab['metal_A'] + df_slab['metal_B']

#Pivot slab and adsorbate energies
dataset_wide = dataset_long.pivot_table(
    index=['metal_A', 'metal_B', 'metal_group'],
    columns='intermediate',
    values='energy_slab_adsorbate'
).reset_index()
dataset_wide.columns.name = None
dataset_wide = dataset_wide.rename(columns={
    'H': 'energy_slab_adsorbate_H',
    'O': 'energy_slab_adsorbate_O',
    'OH': 'energy_slab_adsorbate_OH',
    'OOH': 'energy_slab_adsorbate_OOH'
})

#Merge slab and adsorbate energies with slab energies 
dataset = dataset_wide.merge(
    df_slab[['metal_group', 'energy_slab']],
    on='metal_group',
    how='inner'
)

adsorbate_cols = ['energy_slab_adsorbate_H', 'energy_slab_adsorbate_O',
                  'energy_slab_adsorbate_OH', 'energy_slab_adsorbate_OOH']
n_before = len(dataset)
dataset = dataset.dropna(subset=adsorbate_cols + ['energy_slab']).reset_index(drop=True)
n_dropped = n_before - len(dataset)

#Compute adsorption energies (ΔE)
dataset['delta_E_H'] = dataset['energy_slab_adsorbate_H'] - dataset['energy_slab'] - 0.5 * E_H2
dataset['delta_E_O'] = dataset['energy_slab_adsorbate_O'] - dataset['energy_slab'] - (E_H2O - E_H2)
dataset['delta_E_OH'] = dataset['energy_slab_adsorbate_OH'] - dataset['energy_slab'] - (E_H2O - 0.5 * E_H2)
dataset['delta_E_OOH'] = dataset['energy_slab_adsorbate_OOH'] - dataset['energy_slab'] - (2 * E_H2O - 1.5 * E_H2)

#Compute Gibbs free energies (ΔG)
dataset['delta_G_H'] = dataset['delta_E_H'] + ZPE_H
dataset['delta_G_OH'] = dataset['delta_E_OH'] + ZPE_OH
dataset['delta_G_O'] = dataset['delta_E_O'] + ZPE_O
dataset['delta_G_OOH'] = dataset['delta_E_OOH'] + ZPE_OOH

#Compute OER overpotential (eta_OER)
    #Each step transfers one electron, so a free energy in eV is numerically equal to a potential in V
dataset['delta_G1'] = dataset['delta_G_OH']
dataset['delta_G2'] = dataset['delta_G_O'] - dataset['delta_G_OH']
dataset['delta_G3'] = dataset['delta_G_OOH'] - dataset['delta_G_O']
dataset['delta_G4'] = 4.92 - dataset['delta_G_OOH']
dataset['eta_OER'] = dataset[['delta_G1','delta_G2','delta_G3','delta_G4']].max(axis=1) - 1.23

#Reorder columns
col_order = (
    ['metal_A', 'metal_B', 'metal_group'] +
    ['energy_slab'] +
    ['energy_slab_adsorbate_H', 'energy_slab_adsorbate_O',
     'energy_slab_adsorbate_OH', 'energy_slab_adsorbate_OOH'] +
    ['delta_E_H', 'delta_E_O', 'delta_E_OH', 'delta_E_OOH'] +
    ['delta_G_H', 'delta_G_OH', 'delta_G_O', 'delta_G_OOH'] +
    ['delta_G1', 'delta_G2', 'delta_G3', 'delta_G4', 'eta_OER']
)
dataset = dataset[col_order]

#Export dataset
dataset.to_csv(output_file, index=False)

# 3. Descriptor Engineering

In [ ]:
#Descriptor libraries:
    #Mendeleev documentation - https://mendeleev.readthedocs.io
    #Matminer documentation - https://matminer.readthedocs.io

#Define functions
md = MagpieData()
def get_max_ox_state(symbol):
    """Return maximum oxidation state"""
    ox = element(symbol).oxistates
    return max(ox) if ox else 0
def get_atomic_descriptors(symbol):
    """Return all descriptors for a given element"""
    el = element(symbol)
    mg = Element(symbol)
    ea = el.electron_affinity if el.electron_affinity is not None else 0.0
    return {
        #Descriptor set:
            #Single descriptors
        
        #Geometric
        'atomic_radius': el.atomic_radius_rahm, #Atomic radius 
        'covalent_radius': el.covalent_radius_pyykko, #Covalent radius
        'metallic_radius': el.metallic_radius, #Metallic radius
        'van_der_waals_radius': el.vdw_radius, #Van der Waals radius 
        'miracle_radius': md.get_elemental_property(mg, 'MiracleRadius'), #Miracle radius
        's_zunger_radius': md.get_elemental_property(mg, 'ZungerPP-r_s'), #s-orbital pseudopotential radius
        'p_zunger_radius': md.get_elemental_property(mg, 'ZungerPP-r_p'), #p-orbital pseudopotential radius
        'd_zunger_radius': md.get_elemental_property(mg, 'ZungerPP-r_d'), #d-orbital pseudopotential radius
        'pi_zunger_radius': md.get_elemental_property(mg, 'ZungerPP-r_pi'), #Pi pseudopotential radius
        'sigma_zunger_radius': md.get_elemental_property(mg, 'ZungerPP-r_sigma'), #Sigma pseudopotential radius

        #Electronic
        'valence_electrons': md.get_elemental_property(mg, 'NValence'), #Total valence electrons
        's_valence_electrons': md.get_elemental_property(mg, 'NsValence'), #s-orbital valence electrons
        'p_valence_electrons': md.get_elemental_property(mg, 'NpValence'), #p-orbital valence electrons
        'd_valence_electrons': md.get_elemental_property(mg, 'NdValence'), #d-orbital valence electrons
        'f_valence_electrons': md.get_elemental_property(mg, 'NfValence'), #f-orbital valence electrons
        'total_unfilled_orbitals': md.get_elemental_property(mg, 'NUnfilled'), #Total empty orbitals
        's_unfilled_orbitals': md.get_elemental_property(mg, 'NsUnfilled'), #Empty s-orbitals
        'p_unfilled_orbitals': md.get_elemental_property(mg, 'NpUnfilled'), #Empty p-orbitals
        'd_unfilled_orbitals': md.get_elemental_property(mg, 'NdUnfilled'), #Empty d-orbitals
        'f_unfilled_orbitals': md.get_elemental_property(mg, 'NfUnfilled'), #Empty f-orbitals
        'maximum_oxidation_state': get_max_ox_state(symbol), #Maximum oxidation state
        'minimum_oxidation_state': mg.min_oxidation_state, #Minimum oxidation state
        'pauling_electronegativity': el.en_pauling, #Pauling electronegativity
        'allen_electronegativity': el.en_allen, #Allen electronegativity
        'miedema_electronegativity': el.en_miedema, #Miedema electronegativity
        'miedema_electron_density': md.get_elemental_property(mg, 'n_ws^third'), #Miedema cube root of Wigner-Seitz electron density
        'miedema_phi': md.get_elemental_property(mg, 'phi'), #Miedema phi
        'electron_affinity': ea, #Electron affinity
        'first_ionisation_energy': el.ionenergies.get(1, None), #First ionisation energy
        'second_ionisation_energy': el.ionenergies.get(2, None), #Second ionisation energy
        'third_ionisation_energy': el.ionenergies.get(3, None), #Third ionisation energy
        'dipole_polarisability': el.dipole_polarizability, #Dipole polarisability
        
        #Energetic
        'ground_state_energy_per_atom': md.get_elemental_property(mg, 'GSenergy_pa'), #Ground-state energy per atom
        'ground_state_volume_per_atom': md.get_elemental_property(mg, 'GSvolume_pa'), #Ground-state volume per atom
        'ground_state_bandgap': md.get_elemental_property(mg, 'GSbandgap'), #Ground-state band gap
        'ground_state_magnetic_moment': md.get_elemental_property(mg, 'GSmagmom'), #Ground-state magnetic moment
        'heat_of_formation': el.heat_of_formation, #Heat of formation
        'heat_of_fusion': md.get_elemental_property(mg, 'HeatFusion'), #Heat of fusion
        'heat_of_vaporisation': md.get_elemental_property(mg, 'HeatVaporization'), #Heat of vaporisation
        'melting_point': el.melting_point, #Melting point
        'boiling_point': el.boiling_point, #Boiling point
        'molar_heat_capacity': md.get_elemental_property(mg, 'HeatCapacityMolar'), #Molar heat capacity

        #Positional 
        'atomic_number': el.atomic_number, #Atomic number
        'atomic_weight': el.atomic_weight, #Atomic weight
        'group': el.group_id, #Group 
        'period': el.period, #Period 
        'glawe_number': el.glawe_number, #Glawe number
        'pettifor_number': el.pettifor_number, #Pettifor number
        'mendeleev_number': el.mendeleev_number, #Mendeleev number

            #Pairwise descriptors
    }
def get_pairwise_descriptors(symbol_A, symbol_B, descriptor_lookup):
    """Compute sum, absolute difference, and product pairwise descriptors"""
    PAIRWISE_PROPS = [
        #Geometric
        'atomic_radius',
        'covalent_radius',
        'metallic_radius',
        'van_der_waals_radius',
        'miracle_radius',
        's_zunger_radius',
        'p_zunger_radius',
        'd_zunger_radius',
        'pi_zunger_radius',
        'sigma_zunger_radius',

        #Electronic
        'valence_electrons',
        's_valence_electrons',
        'p_valence_electrons',
        'd_valence_electrons',
        'f_valence_electrons',
        'total_unfilled_orbitals',
        's_unfilled_orbitals',
        'p_unfilled_orbitals',
        'd_unfilled_orbitals',
        'f_unfilled_orbitals',
        'maximum_oxidation_state',
        'minimum_oxidation_state',
        'pauling_electronegativity',
        'allen_electronegativity',
        'miedema_electronegativity',
        'miedema_electron_density',
        'miedema_phi',
        'electron_affinity',
        'first_ionisation_energy',
        'second_ionisation_energy',
        'third_ionisation_energy',
        'dipole_polarisability',

        #Energetic
        'ground_state_energy_per_atom',
        'ground_state_volume_per_atom',
        'ground_state_bandgap',
        'ground_state_magnetic_moment',
        'heat_of_formation',
        'heat_of_fusion',
        'heat_of_vaporisation',
        'melting_point',
        'boiling_point',
        'molar_heat_capacity',

        #Positional
        'atomic_number',
        'atomic_weight',
        'group',
        'period',
        'glawe_number',
        'pettifor_number',
        'mendeleev_number',
]
    dA = descriptor_lookup[symbol_A]
    dB = descriptor_lookup[symbol_B]
    pairwise = {}
    for prop in PAIRWISE_PROPS:
        a = dA[prop]
        b = dB[prop]
        pairwise[f'sum_{prop}'] = a + b
        pairwise[f'difference_{prop}'] = abs(a - b)
        pairwise[f'product_{prop}'] = a * b
    return pairwise

#Build descriptor lookup table
all_metals = pd.concat([dataset['metal_A'], dataset['metal_B']]).unique()
descriptor_lookup = {}
errors = []
for metal in sorted(all_metals):
    try:
        descriptor_lookup[metal] = get_atomic_descriptors(metal)
    except Exception as e:
        errors.append(f" {metal}: Error - {e}")
if errors:
    print("Warning: Errors during descriptor lookup in Mendeleev and Matminer libraries:")
    for err in errors:
        print(err)
else:
    print(f"Descriptor lookup in Mendeleev and Matminer libraries: {len(descriptor_lookup)} metals present")

#Map single descriptors onto dataset
desc_A = dataset['metal_A'].map(descriptor_lookup).apply(pd.Series).add_prefix('metal_A_')
desc_B = dataset['metal_B'].map(descriptor_lookup).apply(pd.Series).add_prefix('metal_B_')

#Compute pairwise descriptors
pairwise_rows = [
    get_pairwise_descriptors(row['metal_A'], row['metal_B'], descriptor_lookup)
    for _, row in dataset.iterrows()
]
desc_pair = pd.DataFrame(pairwise_rows, index=dataset.index)

#Concatenate descriptors onto dataset
dataset_descriptors = pd.concat([dataset, desc_A, desc_B, desc_pair], axis=1)

#Export dataset
descriptor_output_file = os.path.join(base_dir, "dataset_descriptors.csv")
dataset_descriptors.to_csv(descriptor_output_file, index=False)

desc_cols = [c for c in dataset_descriptors.columns if c not in dataset.columns]

# 4. Hydrogen Evolution Reaction

## 4.1. Descriptor Selection

### 4.1.1. Train/Test Split

In [ ]:
#Train/test split in an 80/20 ratio
    #Train/test split performed before descriptor selection to prevent data leakage
    #Random seed set to ensure reproducibility

#Define target and file path
    #Target is delta_G_H for the HER and eta_OER for the OER
target_col = 'delta_G_H'
reaction = 'HER' if target_col == 'delta_G_H' else 'OER'
units = 'eV' if target_col == 'delta_G_H' else 'V'
desktop_output = os.path.join(base_dir, reaction)
os.makedirs(desktop_output, exist_ok=True)

#Define train/test split ratio and set random seed
test_size = 0.20
random_state = 42
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(
    dataset_descriptors,
    dataset_descriptors[target_col],
    groups=dataset_descriptors['metal_group']
))

#Partition dataset into train and test dataframes
    #Test set is stored away and not used again until machine learning model evaluation
train_df = dataset_descriptors.iloc[train_idx].copy().reset_index(drop=True)
test_df = dataset_descriptors.iloc[test_idx].copy().reset_index(drop=True)

#Verify no group leakage between train and test sets
group_leakage = len(set(train_df['metal_group']) & set(test_df['metal_group']))
assert group_leakage == 0, "Warning: Group leakage detected"

### 4.1.2. Descriptor Pruning

#### 4.1.2.1. Mutual Information Analysis on Full Descriptor Set

In [ ]:
#Mutual Information analysis
    #Computed on full descriptor set and training data only

#Define full descriptor columns and training data
X_train_desc = train_df[desc_cols].copy()
y_train_mi = train_df[target_col].copy()

#Compute Mutual Information scores against the target on training data only
scores = mutual_info_regression(
    X_train_desc.values,
    y_train_mi.values,
    random_state=42
)
mi_df = pd.Series(scores, index=desc_cols).sort_values(ascending=False)
mi_df = mi_df.to_frame(name='MI_score')

#Export Mutual Information results
mi_df.round(4).to_excel(os.path.join(desktop_output, f'mutual_information_{target_col}.xlsx'))

#### 4.1.2.2. Pearson Correlation Analysis on Full Descriptor Set

In [ ]:
#Pearson Correlation analysis
    #Identify perfectly and highly correlated descriptor pairs (r >= 0.98)
    #For each such pair, the descriptor with the lower Mutual Information score will be pruned
    #Computed on training data only

X_train_corr = train_df[desc_cols].copy()

#Compute Pearson correlation matrix
pearson_corr = X_train_corr.corr(method='pearson')

#Export all pairs
all_pairs = []
cols = pearson_corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = pearson_corr.iloc[i, j]
        all_pairs.append({
            'Feature_1': cols[i],
            'Feature_2': cols[j],
            'Correlation': round(r, 4),
        })
pearson_all_pairs = pd.DataFrame(all_pairs).sort_values('Correlation', key=abs, ascending=False)
pearson_all_pairs.to_excel(os.path.join(desktop_output, f'pearson_correlation_{target_col}.xlsx'), index=False)

#Identify perfectly and highly correlated pairs (r >= 0.98)
CORR_THRESH = 0.98
pearson_high = [(cols[i], cols[j])
                for i in range(len(cols)) for j in range(i+1, len(cols))
                if abs(pearson_corr.iloc[i, j]) >= CORR_THRESH]

#### 4.1.2.3. Spearman Correlation Analysis on Full Descriptor Set

In [ ]:
#Spearman Correlation analysis
    #Identify perfectly and highly correlated descriptor pairs (r >= 0.98)
    #For each such pair, the descriptor with the lower Mutual Information score will be pruned
    #Computed on training data only

#Compute Spearman correlation matrix
spearman_corr = X_train_corr.corr(method='spearman')

#Export all pairs
all_pairs = []
cols = spearman_corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = spearman_corr.iloc[i, j]
        all_pairs.append({
            'Feature_1': cols[i],
            'Feature_2': cols[j],
            'Correlation': round(r, 4),
        })
spearman_all_pairs = pd.DataFrame(all_pairs).sort_values('Correlation', key=abs, ascending=False)
spearman_all_pairs.to_excel(os.path.join(desktop_output, f'spearman_correlation_{target_col}.xlsx'), index=False)

#Identify perfectly and highly correlated pairs (p >= 0.98)
spearman_high = [(cols[i], cols[j])
                 for i in range(len(cols)) for j in range(i+1, len(cols))
                 if abs(spearman_corr.iloc[i, j]) >= CORR_THRESH]

#Combine correlated pairs from both Pearson and Spearman and deduplicate by the lower Mutual Information score
high_pairs = list(set(pearson_high) | set(spearman_high))
corr_drops = []
for a, b in high_pairs:

    #Retain the higher Mutual Information descriptor and drop the lower Mutual Information descriptor of the pair
    mi_a = mi_df.loc[a, 'MI_score']
    mi_b = mi_df.loc[b, 'MI_score']
    drop = b if mi_a >= mi_b else a
    if drop not in corr_drops:
        corr_drops.append(drop)

#Apply deduplication to form the descriptor set passed to Boruta
desc_cols_dedup = [c for c in desc_cols if c not in corr_drops]

#### 4.1.2.4. Boruta on Pearson and Spearman Correlation Pruned Descriptor Set

In [ ]:
#Run Boruta on descriptor set
boruta = BorutaPy(
    RandomForestRegressor(n_jobs=-1, random_state=42, max_depth=5),
    n_estimators='auto',
    max_iter=1000,
    random_state=42,
    verbose=1
)
boruta.fit(
    train_df[desc_cols_dedup].values,
    train_df[target_col].values
)

#Identify confirmed, tentative, and rejected descriptor sets
confirmed_mask = boruta.support_
tentative_mask = boruta.support_weak_
desc_cols_confirmed = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup)) if confirmed_mask[i]]
desc_cols_tentative = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup)) if tentative_mask[i]]
desc_cols_rejected = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup))
                      if not confirmed_mask[i] and not tentative_mask[i]]

#Use confirmed descriptors only as working descriptor set
desc_cols_pruned = desc_cols_confirmed.copy()

#Export Boruta results
boruta_df = pd.DataFrame({
    'descriptor': desc_cols_dedup,
    'status': ['Confirmed' if confirmed_mask[i]
               else 'Tentative' if tentative_mask[i]
               else 'Rejected'
               for i in range(len(desc_cols_dedup))],
    'ranking': boruta.ranking_
}).sort_values('ranking')
boruta_df.to_excel(os.path.join(desktop_output, f'boruta_{target_col}.xlsx'), index=False)

#### 4.1.2.5. Final Descriptor Set

In [ ]:
#Export final descriptor dataset
dataset_descriptors_pruned = dataset_descriptors[
    ['metal_A', 'metal_B', 'metal_group',
     'energy_slab', 'energy_slab_adsorbate_H', 'energy_slab_adsorbate_O',
     'energy_slab_adsorbate_OH', 'energy_slab_adsorbate_OOH',
     'delta_E_H', 'delta_E_O', 'delta_E_OH', 'delta_E_OOH',
     'delta_G_H', 'delta_G_OH', 'delta_G_O', 'delta_G_OOH',
     'delta_G1', 'delta_G2', 'delta_G3', 'delta_G4', 'eta_OER'] + desc_cols_pruned]
dataset_descriptors_pruned.to_csv(
    os.path.join(desktop_output, f'dataset_descriptors_pruned_{target_col}.csv'), index=False)

#### 4.1.2.6. Pearson and Spearman Correlation and Mutual Information Analysis on Final Descriptor Set

In [ ]:
#Pearson and Spearman Correlation and Mutual Information analysis on final descriptor set
    #Computed on final descriptor set and training data only

X_train_final = train_df[desc_cols_pruned]

#Pearson and Spearman correlation of every descriptor pair
for method in ['pearson', 'spearman']:
    corr = X_train_final.corr(method=method)
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
    pairs = (upper.rename('correlation').rename_axis(['descriptor_1', 'descriptor_2']).reset_index()
             .assign(abs_correlation=lambda d: d['correlation'].abs())
             .sort_values('abs_correlation', ascending=False))
    pairs.to_excel(os.path.join(desktop_output, f'{method}_correlation_{target_col}_final.xlsx'), index=False)

#Mutual Information scores against the target on training data only
scores = mutual_info_regression(
    X_train_final.values,
    train_df[target_col].values,
    random_state=42
)
mi_df_final = pd.Series(scores, index=desc_cols_pruned).sort_values(ascending=False)
mi_df_final = mi_df_final.to_frame(name='MI_score')
mi_df_final.round(4).to_excel(os.path.join(desktop_output, f'mutual_information_{target_col}_final.xlsx'))

## 4.2. Data Pre-Processing

In [ ]:
#Reload dataset
dataset_descriptors_pruned = pd.read_csv(os.path.join(desktop_output, f'dataset_descriptors_pruned_{target_col}.csv'))
train_df = dataset_descriptors_pruned.iloc[train_idx].copy().reset_index(drop=True)
test_df = dataset_descriptors_pruned.iloc[test_idx].copy().reset_index(drop=True)

#Define X (descriptors) and y (target)
X_train = train_df[desc_cols_pruned].values
X_test = test_df[desc_cols_pruned].values
y_train = train_df[target_col].values
y_test = test_df[target_col].values
groups_train = train_df['metal_group'].values
groups_test = test_df['metal_group'].values

#Descriptor scaling
    #Scaler fit on training data only, then applied to test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#Establish SHAP configuration, which is applied uniformly to all models via KernelExplainer (excluding PySR)
    #Background = 50 k-means centroids of the training set 
    #nsamples = 200 coalitions per explanation
    #l1_reg = 0 so that importance is estimated for all descriptors
np.random.seed(42)
shap_background = shap.kmeans(X_train_scaled, 50)
shap_nsamples = 200
shap_l1_reg = 0

#Initialise results storage
results = []
all_predictions = pd.DataFrame()
shap_results = pd.DataFrame()
hyperparam_configs = pd.DataFrame()

## 4.3. Baseline Model

### 4.3.1. Mean Predictor

In [ ]:
#Mean Predictor
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html

#Define cross-validation strategy
outer_cv = GroupKFold(n_splits=5)

#Fit Mean Predictor
mean_pred = DummyRegressor(strategy='mean')
mean_pred.fit(X_train_scaled, y_train)

#Cross-validation using GroupKFold(5)
cv_scores_mean = cross_val_score(
    DummyRegressor(strategy='mean'), X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_mean = -cv_scores_mean.mean()
cv_rmse_std_mean = cv_scores_mean.std()

#Mean Predictor predictions
y_pred_train_mean = mean_pred.predict(X_train_scaled)
y_pred_test_mean = mean_pred.predict(X_test_scaled)

#Mean Predictor performance metrics
train_r2_mean = r2_score(y_train, y_pred_train_mean)
test_r2_mean = r2_score(y_test, y_pred_test_mean)
train_rmse_mean = root_mean_squared_error(y_train, y_pred_train_mean)
test_rmse_mean = root_mean_squared_error(y_test, y_pred_test_mean)
train_mae_mean = mean_absolute_error(y_train, y_pred_train_mean)
test_mae_mean = mean_absolute_error(y_test, y_pred_test_mean)

#Store Mean Predictor results
results.append({
    'model': 'MP',
    'cv_rmse': round(cv_rmse_mean, 4),
    'cv_rmse_std': round(cv_rmse_std_mean, 4),
    'train_r2': round(train_r2_mean, 4),
    'test_r2': round(test_r2_mean, 4),
    'train_rmse': round(train_rmse_mean, 4),
    'test_rmse': round(test_rmse_mean, 4),
    'train_mae': round(train_mae_mean, 4),
    'test_mae': round(test_mae_mean, 4),
    'best_params': 'strategy=mean'
})

#Store Mean Predictor predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'MP',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_mean
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'MP',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_mean
})], ignore_index=True)

#Record Mean Predictor results for Rashomon set
cv_df_mp = pd.DataFrame([{'model': 'MP', 'cv_rmse': cv_rmse_mean,
                          'cv_rmse_std': cv_rmse_std_mean}])
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_mp], ignore_index=True)

## 4.4. Machine Learning Models

### 4.4.1. Linear Regression and SHapley Additive exPlanations

In [ ]:
#Linear Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

#Define cross-validation strategy
outer_cv = GroupKFold(n_splits=5)

#Fit Linear Regression on full training set
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

#Cross-validation using GroupKFold(5)
cv_scores_lr = cross_val_score(
    lr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_lr = -cv_scores_lr.mean()
cv_rmse_std_lr = cv_scores_lr.std()

#Linear Regression predictions
y_pred_train_lr = lr.predict(X_train_scaled)
y_pred_test_lr = lr.predict(X_test_scaled)

#Linear Regression performance metrics
train_r2_lr = r2_score(y_train, y_pred_train_lr)
test_r2_lr = r2_score(y_test, y_pred_test_lr)
train_rmse_lr = root_mean_squared_error(y_train, y_pred_train_lr)
test_rmse_lr = root_mean_squared_error(y_test, y_pred_test_lr)
train_mae_lr = mean_absolute_error(y_train, y_pred_train_lr)
test_mae_lr = mean_absolute_error(y_test, y_pred_test_lr)

#Store Linear Regression hyperparameters
lr_params = {
    'fit_intercept': lr.fit_intercept,
    'positive': lr.positive,
}

#Store Linear Regression results
results.append({
    'model': 'LR',
    'cv_rmse': round(cv_rmse_lr, 4),
    'cv_rmse_std': round(cv_rmse_std_lr, 4),
    'train_r2': round(train_r2_lr, 4),
    'test_r2': round(test_r2_lr, 4),
    'train_rmse': round(train_rmse_lr, 4),
    'test_rmse': round(test_rmse_lr, 4),
    'train_mae': round(train_mae_lr, 4),
    'test_mae': round(test_mae_lr, 4),
    'best_params': str(lr_params)
})

#Store Linear Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'LR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_lr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'LR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_lr
})], ignore_index=True)

#SHAP on Linear Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_lr = shap.KernelExplainer(lr.predict, shap_background)
shap_values_lr = explainer_lr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Linear Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'LR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_lr).mean(axis=0)
})], ignore_index=True)

#Record Linear Regression results for Rashomon set
cv_df_lr = pd.DataFrame([{'model': 'LR', 'cv_rmse': cv_rmse_lr,
                          'cv_rmse_std': cv_rmse_std_lr}])
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_lr], ignore_index=True)

### 4.4.2. Random Forest Regression and SHapley Additive exPlanations

In [ ]:
#Random Forest Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

#Default parameters:    
    #n_estimators = 100    
    #max_depth = None    
    #min_samples_split = 2    
    #min_samples_leaf = 1    
    #max_features = 1.0

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_rfr = {
    'n_estimators': randint(100, 1500),
    'max_depth': [None, 4, 6, 8, 10, 12, 15, 20, 30, 40],
    'min_samples_split': randint(2, 21),
    'min_samples_leaf': randint(1, 11),
    'max_features': ['sqrt', 'log2', 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, None],
    'bootstrap': [True, False],
    'criterion': ['squared_error', 'absolute_error'],
    'min_weight_fraction_leaf': [0.0, 0.01, 0.02, 0.05],
    'max_leaf_nodes': [None, 20, 50, 100, 200],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_rfr = 1000
search_rfr = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist_rfr,
    n_iter=n_iter_rfr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)

search_rfr.fit(X_train_scaled, y_train, groups=groups_train)
rfr = search_rfr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_rfr = cross_val_score(
    rfr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_rfr = -cv_scores_rfr.mean()
cv_rmse_std_rfr = cv_scores_rfr.std()

#Random Forest Regression predictions
y_pred_train_rfr = rfr.predict(X_train_scaled)
y_pred_test_rfr = rfr.predict(X_test_scaled)

#Random Forest Regression performance metrics
train_r2_rfr = r2_score(y_train, y_pred_train_rfr)
test_r2_rfr = r2_score(y_test, y_pred_test_rfr)
train_rmse_rfr = root_mean_squared_error(y_train, y_pred_train_rfr)
test_rmse_rfr = root_mean_squared_error(y_test, y_pred_test_rfr)
train_mae_rfr = mean_absolute_error(y_train, y_pred_train_rfr)
test_mae_rfr = mean_absolute_error(y_test, y_pred_test_rfr)

#Store best Random Forest Regression hyperparameters
rfr_params = {
    'n_estimators': rfr.n_estimators,
    'max_depth': rfr.max_depth,
    'min_samples_split': rfr.min_samples_split,
    'min_samples_leaf': rfr.min_samples_leaf,
    'max_features': rfr.max_features,
    'bootstrap': rfr.bootstrap,
    'criterion': rfr.criterion,
    'min_weight_fraction_leaf': rfr.min_weight_fraction_leaf,
    'max_leaf_nodes': rfr.max_leaf_nodes,
}

#Store Random Forest Regression results
results.append({
    'model': 'RFR',
    'cv_rmse': round(cv_rmse_rfr, 4),
    'cv_rmse_std': round(cv_rmse_std_rfr, 4),
    'train_r2': round(train_r2_rfr, 4),
    'test_r2': round(test_r2_rfr, 4),
    'train_rmse': round(train_rmse_rfr, 4),
    'test_rmse': round(test_rmse_rfr, 4),
    'train_mae': round(train_mae_rfr, 4),
    'test_mae': round(test_mae_rfr, 4),
    'best_params': str(rfr_params)
})

#Store Random Forest Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'RFR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_rfr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'RFR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_rfr
})], ignore_index=True)

#SHAP on Random Forest Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_rfr = shap.KernelExplainer(rfr.predict, shap_background)
shap_values_rfr = explainer_rfr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Random Forest Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'RFR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_rfr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Random Forest Regression hyperparameter configurations for Rashomon set
cv_df_rfr = pd.DataFrame(search_rfr.cv_results_['params'])
cv_df_rfr.insert(0, 'model', 'RFR')
cv_df_rfr['cv_rmse'] = -search_rfr.cv_results_['mean_test_score']
cv_df_rfr['cv_rmse_std'] = search_rfr.cv_results_['std_test_score']
cv_df_rfr = cv_df_rfr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_rfr], ignore_index=True)

### 4.4.3. Gradient Boosting Regression and SHapley Additive exPlanations

In [ ]:
#Gradient Boosting Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html

#Default parameters:
    #learning_rate = 0.1
    #n_estimators = 100
    #max_depth = 3
    #subsample = 1.0
    #max_features = None
    #min_samples_split = 2
    #min_samples_leaf = 1

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_gbr = {
    'learning_rate': uniform(0.01, 0.29),         
    'n_estimators': randint(100, 1000),
    'max_depth': [2, 3, 4, 5, 6],
    'subsample': uniform(0.5, 0.5),               
    'max_features': ['sqrt', 'log2', 0.3, 0.4, 0.5, 0.6, 0.7, None],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5),
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested 
n_iter_gbr = 1000
search_gbr = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gbr,
    n_iter=n_iter_gbr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_gbr.fit(X_train_scaled, y_train, groups=groups_train)
gbr = search_gbr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_gbr = cross_val_score(
    gbr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_gbr = -cv_scores_gbr.mean()
cv_rmse_std_gbr = cv_scores_gbr.std()

#Gradient Boosting Regression predictions
y_pred_train_gbr = gbr.predict(X_train_scaled)
y_pred_test_gbr = gbr.predict(X_test_scaled)

#Gradient Boosting Regression performance metrics
train_r2_gbr = r2_score(y_train, y_pred_train_gbr)
test_r2_gbr = r2_score(y_test, y_pred_test_gbr)
train_rmse_gbr = root_mean_squared_error(y_train, y_pred_train_gbr)
test_rmse_gbr = root_mean_squared_error(y_test, y_pred_test_gbr)
train_mae_gbr = mean_absolute_error(y_train, y_pred_train_gbr)
test_mae_gbr = mean_absolute_error(y_test, y_pred_test_gbr)

#Store best Gradient Boosting Regression hyperparameters
gbr_params = {
    'learning_rate': gbr.learning_rate,
    'n_estimators': gbr.n_estimators,
    'max_depth': gbr.max_depth,
    'subsample': gbr.subsample,
    'max_features': gbr.max_features,
    'min_samples_split': gbr.min_samples_split,
    'min_samples_leaf': gbr.min_samples_leaf,
    'criterion': gbr.criterion,
    'loss': gbr.loss,
}

#Store Gradient Boosting Regression results
results.append({
    'model': 'GBR',
    'cv_rmse': round(cv_rmse_gbr, 4),
    'cv_rmse_std': round(cv_rmse_std_gbr, 4),
    'train_r2': round(train_r2_gbr, 4),
    'test_r2': round(test_r2_gbr, 4),
    'train_rmse': round(train_rmse_gbr, 4),
    'test_rmse': round(test_rmse_gbr, 4),
    'train_mae': round(train_mae_gbr, 4),
    'test_mae': round(test_mae_gbr, 4),
    'best_params': str(gbr_params)
})

#Store Gradient Boosting Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'GBR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_gbr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'GBR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_gbr
})], ignore_index=True)

#SHAP on Gradient Boosting Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_gbr = shap.KernelExplainer(gbr.predict, shap_background)
shap_values_gbr = explainer_gbr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Gradient Boosting Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'GBR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_gbr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Gradient Boosting Regression hyperparameter configurations for Rashomon set
cv_df_gbr = pd.DataFrame(search_gbr.cv_results_['params'])
cv_df_gbr.insert(0, 'model', 'GBR')
cv_df_gbr['cv_rmse'] = -search_gbr.cv_results_['mean_test_score']
cv_df_gbr['cv_rmse_std'] = search_gbr.cv_results_['std_test_score']
cv_df_gbr = cv_df_gbr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_gbr], ignore_index=True)

### 4.4.4. Support Vector Regression and SHapley Additive exPlanations

In [ ]:
#Support Vector Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

#Default parameters:
    #C = 1.0
    #epsilon = 0.1
    #kernel = 'rbf'
    #gamma = 'scale'

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_svr = {
    'kernel': ['rbf', 'poly'],
    'C': loguniform(0.1, 1000),
    'epsilon': uniform(0.01, 0.29),
    'gamma': loguniform(1e-4, 1.0),
    'degree': randint(2, 5), #poly only (not relevant for rbf)
    'coef0': uniform(0.0, 1.0), #poly only (not relevant for rbf)
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_svr = 1000
search_svr = RandomizedSearchCV(
    SVR(),
    param_distributions=param_dist_svr,
    n_iter=n_iter_svr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_svr.fit(X_train_scaled, y_train, groups=groups_train)
svr = search_svr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_svr = cross_val_score(
    svr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_svr = -cv_scores_svr.mean()
cv_rmse_std_svr = cv_scores_svr.std()

#Support Vector Regression predictions
y_pred_train_svr = svr.predict(X_train_scaled)
y_pred_test_svr = svr.predict(X_test_scaled)

#Support Vector Regression performance metrics
train_r2_svr = r2_score(y_train, y_pred_train_svr)
test_r2_svr = r2_score(y_test, y_pred_test_svr)
train_rmse_svr = root_mean_squared_error(y_train, y_pred_train_svr)
test_rmse_svr = root_mean_squared_error(y_test, y_pred_test_svr)
train_mae_svr = mean_absolute_error(y_train, y_pred_train_svr)
test_mae_svr = mean_absolute_error(y_test, y_pred_test_svr)

#Store best Support Vector Regression hyperparameters
svr_params = {
    'kernel': svr.kernel,
    'C': svr.C,
    'epsilon': svr.epsilon,
    'gamma': svr.gamma,
    'degree': svr.degree,
    'coef0': svr.coef0,
}

#Store Support Vector Regression results
results.append({
    'model': 'SVR',
    'cv_rmse': round(cv_rmse_svr, 4),
    'cv_rmse_std': round(cv_rmse_std_svr, 4),
    'train_r2': round(train_r2_svr, 4),
    'test_r2': round(test_r2_svr, 4),
    'train_rmse': round(train_rmse_svr, 4),
    'test_rmse': round(test_rmse_svr, 4),
    'train_mae': round(train_mae_svr, 4),
    'test_mae': round(test_mae_svr, 4),
    'best_params': str(svr_params)
})

#Store Support Vector Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'SVR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_svr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'SVR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_svr
})], ignore_index=True)

#SHAP on Support Vector Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_svr = shap.KernelExplainer(svr.predict, shap_background)
shap_values_svr = explainer_svr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Support Vector Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'SVR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_svr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Support Vector Regression hyperparameter configurations for Rashomon set
cv_df_svr = pd.DataFrame(search_svr.cv_results_['params'])
cv_df_svr.insert(0, 'model', 'SVR')
cv_df_svr['cv_rmse'] = -search_svr.cv_results_['mean_test_score']
cv_df_svr['cv_rmse_std'] = search_svr.cv_results_['std_test_score']
cv_df_svr = cv_df_svr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_svr], ignore_index=True)

### 4.4.5. Gaussian Process Regression and SHapley Additive exPlanations

In [ ]:
#Gaussian Process Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html

#Default parameters:
    #kernel = None 
    #alpha = 1e-10
    #n_restarts_optimizer = 0
    #normalize_y = False

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Candidate kernel structures
kernels_gpr = [
    C(1.0) * RBF(1.0) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=0.5) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=1.5) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=2.5) + WhiteKernel(0.1),
    C(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(0.1),
    C(1.0) * RBF(1.0) + C(1.0) * RationalQuadratic(1.0, 1.0) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=1.5) + C(1.0) * RationalQuadratic(1.0, 1.0) + WhiteKernel(0.1),
    C(1.0) * RBF(length_scale=[1.0] * len(desc_cols_pruned)) + WhiteKernel(0.1),
]

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_gpr = {
    'kernel': kernels_gpr,
    'alpha': loguniform(1e-10, 1e-1),
    'n_restarts_optimizer': [0, 1, 2, 3],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_gpr = 1000
search_gpr = RandomizedSearchCV(
    GaussianProcessRegressor(normalize_y=True, random_state=42),
    param_distributions=param_dist_gpr,
    n_iter=n_iter_gpr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_gpr.fit(X_train_scaled, y_train, groups=groups_train)
gpr = search_gpr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_gpr = cross_val_score(
    gpr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_gpr = -cv_scores_gpr.mean()
cv_rmse_std_gpr = cv_scores_gpr.std()

#Gaussian Process Regression predictions
y_pred_train_gpr = gpr.predict(X_train_scaled)
y_pred_test_gpr = gpr.predict(X_test_scaled)

#Gaussian Process Regression performance metrics
train_r2_gpr = r2_score(y_train, y_pred_train_gpr)
test_r2_gpr = r2_score(y_test, y_pred_test_gpr)
train_rmse_gpr = root_mean_squared_error(y_train, y_pred_train_gpr)
test_rmse_gpr = root_mean_squared_error(y_test, y_pred_test_gpr)
train_mae_gpr = mean_absolute_error(y_train, y_pred_train_gpr)
test_mae_gpr = mean_absolute_error(y_test, y_pred_test_gpr)

#Store best Gaussian Process Regression hyperparameters
gpr_params = {
    'kernel': str(gpr.kernel_),
    'alpha': gpr.alpha,
    'n_restarts_optimizer': gpr.n_restarts_optimizer,
    'normalize_y': gpr.normalize_y,
}

#Store Gaussian Process Regression results
results.append({
    'model': 'GPR',
    'cv_rmse': round(cv_rmse_gpr, 4),
    'cv_rmse_std': round(cv_rmse_std_gpr, 4),
    'train_r2': round(train_r2_gpr, 4),
    'test_r2': round(test_r2_gpr, 4),
    'train_rmse': round(train_rmse_gpr, 4),
    'test_rmse': round(test_rmse_gpr, 4),
    'train_mae': round(train_mae_gpr, 4),
    'test_mae': round(test_mae_gpr, 4),
    'best_params': str(gpr_params)
})

#Store Gaussian Process Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'GPR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_gpr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'GPR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_gpr
})], ignore_index=True)

#SHAP on Gaussian Process Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_gpr = shap.KernelExplainer(gpr.predict, shap_background)
shap_values_gpr = explainer_gpr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Gaussian Process Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'GPR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_gpr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Gaussian Process Regression hyperparameter configurations for Rashomon set
cv_df_gpr = pd.DataFrame(search_gpr.cv_results_['params'])
cv_df_gpr['kernel'] = cv_df_gpr['kernel'].astype(str)
cv_df_gpr.insert(0, 'model', 'GPR')
cv_df_gpr['cv_rmse'] = -search_gpr.cv_results_['mean_test_score']
cv_df_gpr['cv_rmse_std'] = search_gpr.cv_results_['std_test_score']
cv_df_gpr = cv_df_gpr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_gpr], ignore_index=True)

### 4.4.6. Multilayer Perceptron and SHapley Additive exPlanations

In [ ]:
#Multilayer Perceptron
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html

#Default parameters:
    #hidden_layer_sizes = (100,)
    #activation = 'relu'
    #solver = 'adam'
    #alpha = 0.0001
    #learning_rate_init = 0.001
    #max_iter = 200

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_mlp = {
    'hidden_layer_sizes': [(32,), (64,), (128,),
                           (32, 16), (64, 32), (128, 64), (64, 32, 16)],
    'activation': ['relu', 'tanh'],
    'alpha': loguniform(1e-5, 3),        
    'learning_rate_init': loguniform(1e-4, 1e-1),
    'solver': ['adam', 'lbfgs'],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested 
n_iter_mlp = 1000
search_mlp = RandomizedSearchCV(
    MLPRegressor(random_state=42, max_iter=2000, early_stopping=False),
    param_distributions=param_dist_mlp,
    n_iter=n_iter_mlp,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_mlp.fit(X_train_scaled, y_train, groups=groups_train)
mlp = search_mlp.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_mlp = cross_val_score(
    mlp, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_mlp = -cv_scores_mlp.mean()
cv_rmse_std_mlp = cv_scores_mlp.std()

#Multilayer Perceptron predictions
y_pred_train_mlp = mlp.predict(X_train_scaled)
y_pred_test_mlp = mlp.predict(X_test_scaled)

#Multilayer Perceptron performance metrics
train_r2_mlp = r2_score(y_train, y_pred_train_mlp)
test_r2_mlp = r2_score(y_test, y_pred_test_mlp)
train_rmse_mlp = root_mean_squared_error(y_train, y_pred_train_mlp)
test_rmse_mlp = root_mean_squared_error(y_test, y_pred_test_mlp)
train_mae_mlp = mean_absolute_error(y_train, y_pred_train_mlp)
test_mae_mlp = mean_absolute_error(y_test, y_pred_test_mlp)

#Store best Multilayer Perceptron hyperparameters
mlp_params = {
    'hidden_layer_sizes': mlp.hidden_layer_sizes,
    'activation': mlp.activation,
    'solver': mlp.solver,
    'alpha': mlp.alpha,
    'learning_rate_init': mlp.learning_rate_init,
    'max_iter': mlp.max_iter,
}

#Store Multilayer Perceptron results
results.append({
    'model': 'MLP',
    'cv_rmse': round(cv_rmse_mlp, 4),
    'cv_rmse_std': round(cv_rmse_std_mlp, 4),
    'train_r2': round(train_r2_mlp, 4),
    'test_r2': round(test_r2_mlp, 4),
    'train_rmse': round(train_rmse_mlp, 4),
    'test_rmse': round(test_rmse_mlp, 4),
    'train_mae': round(train_mae_mlp, 4),
    'test_mae': round(test_mae_mlp, 4),
    'best_params': str(mlp_params)
})

#Store Multilayer Perceptron predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'MLP',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_mlp
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'MLP',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_mlp
})], ignore_index=True)

#SHAP on Multilayer Perceptron — KernelExplainer on the test set
np.random.seed(42)
explainer_mlp = shap.KernelExplainer(mlp.predict, shap_background)
shap_values_mlp = explainer_mlp.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Multilayer Perceptron SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'MLP',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_mlp).mean(axis=0)
})], ignore_index=True)

#Record all sampled Multilayer Perceptron hyperparameter configurations for Rashomon set
cv_df_mlp = pd.DataFrame(search_mlp.cv_results_['params'])
cv_df_mlp.insert(0, 'model', 'MLP')
cv_df_mlp['cv_rmse'] = -search_mlp.cv_results_['mean_test_score']
cv_df_mlp['cv_rmse_std'] = search_mlp.cv_results_['std_test_score']
cv_df_mlp = cv_df_mlp.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_mlp], ignore_index=True)

### 4.4.7. Python Symbolic Regression

In [ ]:
#PySR
    #Implementation: https://github.com/MilesCranmer/PySR

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#PySR configuration
pysr_base = dict(
    niterations=200,
    binary_operators=["+", "-", "*"],
    unary_operators=["square", "cube", "abs"],
    populations=30,
    population_size=50,
    constraints={"square": 9, "cube": 6},
    nested_constraints={"square": {"square": 0, "cube": 0},
                        "cube":   {"square": 0, "cube": 0}},
    model_selection="best",
    random_state=42,
    progress=False,
    verbosity=0,
)

#Deterministic settings for fits, and parallel settings for tuning search
pysr_deterministic = dict(deterministic=True, procs=0, multithreading=False)
pysr_parallel = dict(deterministic=False, parallelism="multithreading", procs=4)

#Hyperparameter tuning
    #Tuned on the training data only using inner GroupKFold(3)
param_grid_pysr = {
    'maxsize': [15, 20, 25, 30, 35, 40, 50, 60],
    'parsimony': [1e-5, 1e-4, 1e-3, 1e-2],
}

inner_folds = list(inner_cv.split(X_train_scaled, y_train, groups=groups_train))
pysr_cv_records = []

for ms, pars in itertools.product(param_grid_pysr['maxsize'], param_grid_pysr['parsimony']):
    t_cfg = time.time()
    fold_rmse, fold_cplx = [], []
    for tr_i, va_i in inner_folds:
        model_i = PySRRegressor(**{**pysr_base, **pysr_parallel,
                                   'maxsize': ms, 'parsimony': pars})
        model_i.fit(X_train_scaled[tr_i], y_train[tr_i])
        pred_i = model_i.predict(X_train_scaled[va_i])
        n_bad = int((~np.isfinite(pred_i)).sum())
        if n_bad:
            print(f"Warning: {n_bad} non-finite predictions")
            pred_i = np.where(np.isfinite(pred_i), pred_i, y_train[tr_i].mean())
        fold_rmse.append(root_mean_squared_error(y_train[va_i], pred_i))
        fold_cplx.append(int(model_i.get_best()['complexity']))
    pysr_cv_records.append({
        'maxsize': ms,
        'parsimony': pars,
        'cv_rmse': np.mean(fold_rmse),
        'cv_rmse_sd': np.std(fold_rmse),
        'mean_complexity': np.mean(fold_cplx),
        'minutes': (time.time() - t_cfg) / 60,
    })
    r = pysr_cv_records[-1]
    print(f"  maxsize={ms:<3} parsimony={pars:<7} | inner CV RMSE {r['cv_rmse']:.4f} "
          f"± {r['cv_rmse_sd']:.4f} | complexity {r['mean_complexity']:.1f} "
          f"| {r['minutes']:.1f} min")

cv_df_pysr = pd.DataFrame(pysr_cv_records).sort_values('cv_rmse').reset_index(drop=True)

#Select the configuration with the lowest inner CV RMSE
best_cfg = cv_df_pysr.iloc[0]
pysr_maxsize = int(best_cfg['maxsize'])
pysr_parsimony = float(best_cfg['parsimony'])

#Assemble the tuned configuration used for all fits
pysr_kwargs = {**pysr_base, **pysr_deterministic,
               'maxsize': pysr_maxsize, 'parsimony': pysr_parsimony}

#Cross-validation using GroupKFold(5) on the tuned configuration
cv_rmse_folds = []
for fold_i, (tr, va) in enumerate(outer_cv.split(X_train_scaled, y_train, groups=groups_train)):
    model_f = PySRRegressor(**pysr_kwargs)
    model_f.fit(X_train_scaled[tr], y_train[tr])
    pred_va = model_f.predict(X_train_scaled[va])
    cv_rmse_folds.append(root_mean_squared_error(y_train[va], pred_va))

cv_rmse_pysr = np.mean(cv_rmse_folds)
cv_rmse_std_pysr = np.std(cv_rmse_folds)

#Fit final PySR on the full training set
pysr = PySRRegressor(**pysr_kwargs)
pysr.fit(X_train_scaled, y_train)

#PySR predictions (best-selected equation)
y_pred_train_pysr = pysr.predict(X_train_scaled)
y_pred_test_pysr = pysr.predict(X_test_scaled)

#Verify that the selected equation is finite across both the training and test sets
n_nonfinite = int((~np.isfinite(y_pred_train_pysr)).sum() + (~np.isfinite(y_pred_test_pysr)).sum())
if n_nonfinite:
    print(f"Warning: {n_nonfinite} non-finite predictions")

#PySR performance metrics
train_r2_pysr = r2_score(y_train, y_pred_train_pysr)
test_r2_pysr = r2_score(y_test, y_pred_test_pysr)
train_rmse_pysr = root_mean_squared_error(y_train, y_pred_train_pysr)
test_rmse_pysr = root_mean_squared_error(y_test, y_pred_test_pysr)
train_mae_pysr = mean_absolute_error(y_train, y_pred_train_pysr)
test_mae_pysr = mean_absolute_error(y_test, y_pred_test_pysr)

#Store PySR selected equation and tuned parameters
best_eq = pysr.get_best()
pysr_params = {
    'complexity': int(best_eq['complexity']),
    'equation': str(best_eq['equation']),
    'maxsize': pysr_maxsize,
    'parsimony': pysr_parsimony,
    'niterations': pysr_base['niterations'],
    'binary_operators': pysr_base['binary_operators'],
    'unary_operators': pysr_base['unary_operators'],
    'model_selection': pysr_base['model_selection'],
}

#Store PySR results
results.append({
    'model': 'PySR',
    'cv_rmse': round(cv_rmse_pysr, 4),
    'cv_rmse_std': round(cv_rmse_std_pysr, 4),
    'train_r2': round(train_r2_pysr, 4),
    'test_r2': round(test_r2_pysr, 4),
    'train_rmse': round(train_rmse_pysr, 4),
    'test_rmse': round(test_rmse_pysr, 4),
    'train_mae': round(train_mae_pysr, 4),
    'test_mae': round(test_mae_pysr, 4),
    'best_params': str(pysr_params)
})

#Store PySR predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'PySR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_pysr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'PySR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_pysr
})], ignore_index=True)

#Record all PySR hyperparameter configurations for Rashomon set
cv_df_pysr_out = cv_df_pysr.copy()
cv_df_pysr_out.insert(0, 'model', 'PySR')
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_pysr_out], ignore_index=True)

#Export all equations with the best one flagged
eq_export = pysr.equations_.copy()
eq_export['selected'] = eq_export['complexity'] == best_eq['complexity']
eq_export.to_excel(os.path.join(desktop_output, f'pysr_{target_col}_all_equations.xlsx'), index=False)

#Export results of all models (single split) and the hyperparameter configurations
pd.DataFrame(results).to_excel(os.path.join(desktop_output, f'ml_results_{target_col}.xlsx'), index=False)
all_predictions.to_excel(os.path.join(desktop_output, f'ml_predictions_{target_col}.xlsx'), index=False)
shap_results.to_excel(os.path.join(desktop_output, f'shap_results_{target_col}.xlsx'), index=False)
hyperparam_configs.to_excel(os.path.join(desktop_output, f'hyperparam_configs_{target_col}.xlsx'), index=False)

## 4.5. Constructing a Rashomon Set

In [ ]:
#Rashomon set construction
cfg = hyperparam_configs.copy()

if 'cv_rmse_sd' in cfg.columns:
    cfg['cv_rmse_std'] = cfg['cv_rmse_std'].fillna(cfg['cv_rmse_sd'])

#Classify every sampled configuration
scale = y_train.std()
cfg['status'] = np.where(cfg['cv_rmse'].isna(), 'failed',
                  np.where(cfg['cv_rmse'] > 10 * scale, 'diverged', 'usable'))
usable = cfg[cfg['status'] == 'usable'].copy()

#Rashomon set threshold, anchored to the fold-to-fold cross-validation RMSE variability of the best model
    #RASHOMON_SD = number of fold-to-fold standard deviations above the best configuration
    #Increase number of standard deviations to enlarge the Rashomon set, then re-run the construction and refitting
RASHOMON_SD = 1
best = usable.loc[usable['cv_rmse'].idxmin()]
eps = best['cv_rmse_std']
if not np.isfinite(eps):
    eps = usable['cv_rmse_std'].min()
    print("Warning: No fold-to-fold standard deviation for the best model")
rashomon_threshold_fixed = best['cv_rmse'] + RASHOMON_SD * eps
usable['member'] = usable['cv_rmse'] <= rashomon_threshold_fixed

#Candidate pool and Rashomon set members by architecture
ratio_rows = []
for m, g in cfg.groupby('model'):
    u = usable[usable.model == m]
    ratio_rows.append({'reaction': reaction, 'architecture': m,
                       'sampled': len(g), 'usable': len(u),
                       'failed': int((g.status == 'failed').sum()),
                       'diverged': int((g.status == 'diverged').sum()),
                       'members': int(u.member.sum()), 'ratio': u.member.mean(),
                       'best_cv_rmse': u.cv_rmse.min(),
                       'median_cv_rmse': u.cv_rmse.median(),
                       'threshold': rashomon_threshold_fixed, 'units': units})

members_fixed = sorted(usable.loc[usable.member, 'model'].unique(),
                       key=lambda m: usable.loc[usable.model == m, 'cv_rmse'].min())
rashomon_ratio = pd.DataFrame(ratio_rows)
candidate_pool = usable

#Export results
rashomon_ratio.round(4).to_excel(
    os.path.join(desktop_output, f'rashomon_ratio_{target_col}.xlsx'), index=False)
candidate_pool.round(6).to_excel(
    os.path.join(desktop_output, f'candidate_model_pool_{target_col}.xlsx'), index=False)

## 4.6. Refitting the Rashomon Set

In [ ]:
#Refitting the Rashomon set across configurations and training seeds
    #Every member of the Rashomon set is refitted on the same training rows and evaluated on the same test set
    #Models differ from one another only in architecture, hyperparameter configuration, and the random seed used during training
    #Each fitted model is a competing account of the same dataset and three nested sources of variation are represented
    #Models sharing a configuration differ only by training seed; models sharing an architecture differ by configuration; 
    #models of different architectures differ by algorithm

#Number of configurations per architecture (above actual maximum to clear the maximum), and training seeds per configuration
N_CONFIGS_PER_ARCH = 2000
TRAIN_SEEDS = [0, 1, 2]

#Configurations to fit in this run (batched)
BATCH_SIZE = None

#Order in which configurations are fitted
SHUFFLE_SEED = 42

REFIT_PATH = os.path.join(desktop_output, f'rashomon_refit_dac_{target_col}.csv')
SHAP_PATH_FIXED = os.path.join(desktop_output, f'rashomon_refit_shap_{target_col}.csv')

#Estimator constructors
_ESTIMATORS = {
    'RFR': (RandomForestRegressor, {}),
    'GBR': (GradientBoostingRegressor, {}),
    'SVR': (SVR, {}),
    'GPR': (GaussianProcessRegressor, {'normalize_y': True}),
    'MLP': (MLPRegressor, {'max_iter': 2000, 'early_stopping': False}),
    'LR':  (LinearRegression, {}),
}

#Deterministic architectures whose fit does not depend on a random seed
_DETERMINISTIC = {'SVR', 'LR'}

#Kernel architecture
_KERNEL_LOOKUP = {str(k): k for k in kernels_gpr}

def build_estimator(row, seed):
    """Reconstruct one sampled configuration"""
    arch = row['model']
    cls, fixed = _ESTIMATORS[arch]
    params = {c: row[c] for c in candidate_pool.attrs['param_cols']
              if c in row.index and pd.notna(row[c])}
    for p in ('n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf',
              'max_leaf_nodes', 'degree', 'n_restarts_optimizer'):
        if p in params:
            params[p] = int(params[p])
    if arch == 'GPR':
        params['kernel'] = _KERNEL_LOOKUP[params['kernel']]
    if 'random_state' in cls().get_params():
        params['random_state'] = seed
    return cls(**{**fixed, **params})

#Columns of the candidate pool that describe a configuration
candidate_pool.attrs['param_cols'] = [
    c for c in candidate_pool.columns
    if c not in ('model', 'status', 'member', 'cv_rmse', 'cv_rmse_std', 'cv_rmse_sd',
                 'mean_complexity', 'minutes')]

#Select configurations to refit
selected = []
for arch, g in candidate_pool[candidate_pool.member].groupby('model'):
    if arch not in _ESTIMATORS:
        print(f"  {arch} skipped - not reconstructable")
        continue
    g = g.sort_values('cv_rmse')
    take = g if len(g) <= N_CONFIGS_PER_ARCH else g.iloc[
        np.linspace(0, len(g) - 1, N_CONFIGS_PER_ARCH).astype(int)]
    take = take.copy()
    take['config_id'] = [f"{arch}_{i:03d}" for i in range(len(take))]
    selected.append(take)
selected = pd.concat(selected, ignore_index=True)
selected = selected.sample(frac=1, random_state=SHUFFLE_SEED).reset_index(drop=True)

#Resume from whichever configurations are already recorded
def _load(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        return df, set(df['config_id'].unique())
    return pd.DataFrame(), set()

rank_prev, rank_done = _load(REFIT_PATH)
shap_prev, shap_done = _load(SHAP_PATH_FIXED)
done_configs = rank_done & shap_done
rank_records = (rank_prev[rank_prev['config_id'].isin(done_configs)].to_dict('records')
                if len(rank_prev) else [])
shap_records = (shap_prev[shap_prev['config_id'].isin(done_configs)].to_dict('records')
                if len(shap_prev) else [])

#Configurations in this run that still need fitting
batch_todo = selected[~selected['config_id'].isin(done_configs)]
if BATCH_SIZE is not None:
    batch_todo = batch_todo.head(BATCH_SIZE)

#Fixed reference set
ref_groups = test_df['metal_group'].values

#Save with retries
failures = []
def _save(retries=3):
    for attempt in range(retries):
        try:
            pd.DataFrame(rank_records).to_csv(REFIT_PATH, index=False)
            pd.DataFrame(shap_records).to_csv(SHAP_PATH_FIXED, index=False)
            return True
        except OSError as err:
            print(f"Save failed ({type(err).__name__}); retry {attempt + 1}/{retries}")
            time.sleep(5)
    print("Warning: Save failed after retries")
    return False

t_start = time.time()
for n, (_, row) in enumerate(batch_todo.iterrows(), 1):
    arch = row['model']
    seeds = [0] if arch in _DETERMINISTIC else TRAIN_SEEDS
    for s in seeds:
        try:
            model = build_estimator(row, s)
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_test_scaled)

            #DAC rankings
            order = np.argsort(np.abs(pred) if target_col == 'delta_G_H' else pred)
            for r, idx in enumerate(order):
                rank_records.append({'architecture': arch, 'config_id': row['config_id'],
                                     'train_seed': s, 'metal_group': ref_groups[idx],
                                     'pred': pred[idx], 'rank': r + 1})

            #SHAP rankings
            sv = shap.KernelExplainer(model.predict, shap_background).shap_values(
                X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)
            for d, val in zip(desc_cols_pruned, np.abs(sv).mean(axis=0)):
                shap_records.append({'architecture': arch, 'config_id': row['config_id'],
                                     'train_seed': s, 'descriptor': d,
                                     'mean_abs_shap': val})
        except Exception as e:
            failures.append({'architecture': arch, 'config_id': row['config_id'],
                             'train_seed': s, 'error': type(e).__name__})

    #Incremental save
    if n % 20 == 0 or n == len(batch_todo):
        _save()
        el = time.time() - t_start
        print(f"  {n}/{len(batch_todo)} this run | "
              f"{len(done_configs) + n}/{len(selected)} overall | "
              f"{el/60:.1f} min elapsed | ~{(len(batch_todo) - n) * el / n / 60:.1f} min left")

_save()
refit_ranks = pd.DataFrame(rank_records)
refit_shap = pd.DataFrame(shap_records)

#Print completion and any failed fits
print(f"{refit_ranks['config_id'].nunique()}/{len(selected)} configurations complete; {len(failures)} failures this run")
if failures:
    print(pd.DataFrame(failures).groupby(['architecture', 'error']).size().to_string())

## 4.7. Consensus Across the Rashomon Set

### 4.7.1. Agreement Within a Configuration, Within an Architecture, and Across Architectures

In [ ]:
#Consensus across the Rashomon set
    #Agreement measured within a configuration, within an architecture, and across architectures

REFIT_PATH = os.path.join(desktop_output, f'rashomon_refit_dac_{target_col}.csv')
SHAP_PATH_FIXED = os.path.join(desktop_output, f'rashomon_refit_shap_{target_col}.csv')

def consensus_levels(df, item_col, val_col):
    """Spearman agreement within a configuration, within an architecture, and across architectures"""
    wide = (df.drop_duplicates(subset=['architecture', 'config_id', 'train_seed', item_col])
              .pivot_table(index=['architecture', 'config_id', 'train_seed'],
                           columns=item_col, values=val_col)
              .sort_index())
    C = np.corrcoef(rankdata(wide.to_numpy(), axis=1))
    arch = wide.index.get_level_values('architecture').to_numpy()
    cfg = wide.index.get_level_values('config_id').to_numpy()

    iu = np.triu_indices(len(C), k=1)
    same_cfg = (cfg[:, None] == cfg[None, :])[iu]
    same_arch = (arch[:, None] == arch[None, :])[iu]
    rho = C[iu]
    ok = np.isfinite(rho)

    def _stat(mask):
        v = rho[mask & ok]
        return (v.mean(), v.std(), v.size) if v.size else (np.nan, np.nan, 0)

    #Agreement between each pair of architectures, and the mean over those pairs
    archs = sorted(set(arch))
    pair_matrix = pd.DataFrame(np.nan, index=archs, columns=archs, dtype=float)
    pair_means = []
    for i in range(len(archs)):
        for j in range(i + 1, len(archs)):
            m = (((arch[:, None] == archs[i]) & (arch[None, :] == archs[j])) |
                 ((arch[:, None] == archs[j]) & (arch[None, :] == archs[i])))[iu]
            v = rho[m & ok]
            if v.size:
                pair_matrix.iloc[i, j] = pair_matrix.iloc[j, i] = v.mean()
                pair_means.append(v.mean())
    #How much an architecture agrees with itself across configurations
    for i, a in enumerate(archs):
        m = ((arch[:, None] == a) & (arch[None, :] == a))[iu] & ~same_cfg
        v = rho[m & ok]
        if v.size:
            pair_matrix.iloc[i, i] = v.mean()

    #Seed and configuration agreement within each architecture, then averaged over architectures
    seed_by_arch, cfg_by_arch = [], []
    for a in archs:
        m = ((arch[:, None] == a) & (arch[None, :] == a))[iu] & ok
        s, c = rho[m & same_cfg], rho[m & ~same_cfg]
        seed_by_arch.append(s.mean() if s.size else 1.0)   
        cfg_by_arch.append(c.mean() if c.size else np.nan)

    return {'seed': (np.mean(seed_by_arch), np.std(seed_by_arch), len(seed_by_arch)),
            'config': (np.nanmean(cfg_by_arch), np.nanstd(cfg_by_arch), int(np.isfinite(cfg_by_arch).sum())),
            'arch': (np.mean(pair_means), np.std(pair_means), len(pair_means)),
            'arch_unweighted': _stat(~same_arch),
            'pair_matrix': pair_matrix,
            'wide': wide, 'C': C, 'arch_labels': arch, 'cfg_labels': cfg}

results_levels = {}
for axis, path, item_col, val_col in [
        ('DAC Ranking', REFIT_PATH, 'metal_group', 'rank'),
        ('SHAP Attribution', SHAP_PATH_FIXED, 'descriptor', 'mean_abs_shap')]:
    results_levels[axis] = consensus_levels(pd.read_csv(path), item_col, val_col)

#Seed and configuration agreement of each architecture
arch_rows = []
for a in sorted(set(results_levels['DAC Ranking']['arch_labels'])):
    vals = {}
    for axis in results_levels:
        r = results_levels[axis]
        m = r['arch_labels'] == a
        sub = r['C'][np.ix_(m, m)]
        c_sub = r['cfg_labels'][m]
        iu = np.triu_indices(len(sub), k=1)
        same = (c_sub[:, None] == c_sub[None, :])[iu]
        v = sub[iu]
        ok = np.isfinite(v)
        vals[axis] = (v[same & ok].mean() if (same & ok).any() else np.nan,
                      v[~same & ok].mean() if (~same & ok).any() else np.nan)
    d_s, d_c = vals['DAC Ranking']
    s_s, s_c = vals['SHAP Attribution']
    arch_rows.append({'reaction': reaction, 'architecture': a,
                      'dac_seed_rho': d_s, 'dac_config_rho': d_c,
                      'shap_seed_rho': s_s, 'shap_config_rho': s_c})

#Values carried forward
consensus_fixed = pd.DataFrame([
    {'reaction': reaction, 'axis': axis,
     'seed_rho': results_levels[axis]['seed'][0],
     'seed_archs': results_levels[axis]['seed'][2],
     'config_rho': results_levels[axis]['config'][0],
     'config_archs': results_levels[axis]['config'][2],
     'arch_rho': results_levels[axis]['arch'][0],
     'arch_pairs': results_levels[axis]['arch'][2],
     'arch_rho_by_instance': results_levels[axis]['arch_unweighted'][0]}
    for axis in results_levels])
consensus_by_arch = pd.DataFrame(arch_rows)
consensus_pairs = {axis: results_levels[axis]['pair_matrix'] for axis in results_levels}

#Contribution of each source, one row per axis
consensus_contributions = pd.DataFrame([
    {'reaction': reaction, 'axis': axis,
     'training_seed': 1.0 - results_levels[axis]['seed'][0],
     'hyperparameters': results_levels[axis]['seed'][0] - results_levels[axis]['config'][0],
     'architecture': results_levels[axis]['config'][0] - results_levels[axis]['arch'][0],
     'total': 1.0 - results_levels[axis]['arch'][0]}
    for axis in results_levels])

#Export results
with pd.ExcelWriter(os.path.join(
        desktop_output, f'rashomon_consensus_{target_col}.xlsx')) as writer:
    consensus_fixed.round(4).to_excel(writer, sheet_name='Levels', index=False)
    consensus_contributions.round(4).to_excel(writer, sheet_name='Contributions', index=False)
    consensus_by_arch.round(4).to_excel(writer, sheet_name='By-Architecture', index=False)
    for axis, pm in consensus_pairs.items():
        pm.round(4).to_excel(writer, sheet_name=f'Pairs-{axis.split()[0]}')

### 4.7.2. Vote-Based Consensus

In [ ]:
#Vote-based consensus across the Rashomon set
    #Each refitted model votes for its top five DACs and its top five descriptors
    #A DAC or descriptor reaches a majority consensus if its vote share is greater than 50%
    #Proportional weighting counts every refitted model once, so an architecture's influence scales with its number of refits
    #Equal weighting averages the vote share within each architecture, such that every architecture counts once

#Define top N and majority voting criteria
TOP_N = 5 
MAJORITY = 0.50
KEY = ['architecture', 'config_id', 'train_seed']

#Top-ranked DACs and descriptors of every refitted model
top_dac = refit_ranks[refit_ranks['rank'] <= TOP_N]
top_desc = (refit_shap.sort_values('mean_abs_shap', ascending=False)
            .groupby(KEY).head(TOP_N))

#Number of refitted models per architecture
n_models = top_dac[KEY].drop_duplicates().groupby('architecture').size()

#Vote share of every DAC and descriptor under both weightings
votes = {}
for kind, df, col in [('DAC', top_dac, 'metal_group'), ('Descriptor', top_desc, 'descriptor')]:
    per_arch = df.groupby(['architecture', col]).size().unstack(fill_value=0).div(n_models, axis=0)
    table = pd.DataFrame({'proportional': df.groupby(col).size() / n_models.sum(),
                          'equal': per_arch.mean(axis=0)})
    table = table.sort_values('proportional', ascending=False)
    table['majority_proportional'] = table['proportional'] > MAJORITY
    table['majority_equal'] = table['equal'] > MAJORITY
    votes[kind] = table.rename_axis(col)

#Export results
with pd.ExcelWriter(os.path.join(desktop_output, f'rashomon_votes_{target_col}.xlsx')) as writer:
    votes['DAC'].round(4).to_excel(writer, sheet_name='DAC')
    votes['Descriptor'].round(4).to_excel(writer, sheet_name='Descriptor')

### 4.7.3. Agreement on Direction of SHapley Additive exPlanation Effects

In [ ]:
#SHAP direction of each descriptor
    #Spearman correlation between a descriptor's raw value and its signed SHAP value across the test set

#Signed SHAP values of the fixed-split models
signed_shap = {'LR': shap_values_lr, 'RFR': shap_values_rfr, 'GBR': shap_values_gbr,
               'SVR': shap_values_svr, 'GPR': shap_values_gpr, 'MLP': shap_values_mlp}

direction_df = pd.DataFrame([
    {'architecture': arch, 'descriptor': d, 'rho': spearmanr(X_test[:, j], sv[:, j]).correlation}
    for arch, sv in signed_shap.items()
    for j, d in enumerate(desc_cols_pruned)])

#Agreement on SHAP sign 
direction_wide = direction_df.pivot(index='descriptor', columns='architecture', values='rho')[list(signed_shap)]
in_set = direction_wide[[a for a in direction_wide.columns if a in members_fixed]]
direction_wide['unanimous_in_rashomon_set'] = in_set.apply(
    lambda r: bool((np.sign(r) == np.sign(r.iloc[0])).all()), axis=1)

#Signed SHAP values
signed_long = pd.concat([
    pd.DataFrame(sv, columns=desc_cols_pruned).assign(metal_group=test_df['metal_group'].values, architecture=arch)
    for arch, sv in signed_shap.items() if arch in members_fixed
]).melt(id_vars=['metal_group', 'architecture'], var_name='descriptor', value_name='signed_shap')

#Export results
direction_wide.round(4).to_excel(os.path.join(desktop_output, f'shap_direction_{target_col}.xlsx'))
signed_long.round(6).to_excel(os.path.join(desktop_output, f'shap_signed_{target_col}.xlsx'), index=False)

## 4.8. Model Performance Under Resampling

### 4.8.1. MP, LR, RFR, GBR, SVR, GPR, MLP

In [ ]:
#Resampling across 1000 random seeds
seeds = list(range(1000))

#Define X, y, groups from final descriptor set
X = dataset_descriptors_pruned[desc_cols_pruned].values
y = dataset_descriptors_pruned[target_col].values
groups = dataset_descriptors_pruned['metal_group'].values

#Define models
robustness_models = {
    'MP': DummyRegressor(strategy='mean'),
    'LR': LinearRegression(),
    'RFR': clone(rfr),
    'GBR': clone(gbr),
    'SVR': clone(svr),
    'GPR': clone(gpr),
    'MLP': clone(mlp),
}

robustness_models['RFR'].set_params(n_jobs=-1)
    
#Storage of results
robustness_results = []

t_start = time.time()

for seed in seeds:
    #Re-split with seed, test_size = 0.20 
    gss_rob = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, test_idx = next(gss_rob.split(X, y, groups=groups))
    X_train_rob, X_test_rob = X[train_idx], X[test_idx]
    y_train_rob, y_test_rob = y[train_idx], y[test_idx]

    #Scale per seed and fit on train only 
    scaler_rob = StandardScaler()
    X_train_rob_scaled = scaler_rob.fit_transform(X_train_rob)
    X_test_rob_scaled = scaler_rob.transform(X_test_rob)

    for name, model in robustness_models.items():
        try:
            m = clone(model)
            m.fit(X_train_rob_scaled, y_train_rob)
            y_pred_test_rob = m.predict(X_test_rob_scaled)
            y_pred_train_rob = m.predict(X_train_rob_scaled)
            robustness_results.append({
                'seed': seed,
                'model': name,
                'train_rmse': round(root_mean_squared_error(y_train_rob, y_pred_train_rob), 4),
                'test_rmse': round(root_mean_squared_error(y_test_rob, y_pred_test_rob), 4),
                'train_r2': round(r2_score(y_train_rob, y_pred_train_rob), 4),
                'test_r2': round(r2_score(y_test_rob, y_pred_test_rob), 4),
                'train_mae': round(mean_absolute_error(y_train_rob, y_pred_train_rob), 4),
                'test_mae': round(mean_absolute_error(y_test_rob, y_pred_test_rob), 4),
            })
        except Exception as e:
            print(f"Seed {seed} {name}: Failed — {e}")
    
    if (seed + 1) % 10 == 0:
        el = time.time() - t_start
        print(f"Completed {seed + 1}/{len(seeds)} seeds | {el/60:.1f} min elapsed | "
              f"~{(len(seeds) - seed - 1) * el / (seed + 1) / 60:.1f} min left")
    #Incremental save
    if (seed + 1) % 50 == 0:
        pd.DataFrame(robustness_results).to_excel(
            os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'), index=False)

#Export results
pd.DataFrame(robustness_results).to_excel(
    os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'), index=False)

### 4.8.2. PySR

In [ ]:
#Identical PySR configuration except parallel and non-deterministic
pysr_rob_kwargs = {**pysr_base, **pysr_parallel,
                   'maxsize': pysr_maxsize, 'parsimony': pysr_parsimony}

#Seeds to run 
PYSR_SEEDS = list(range(0, 1000))
TOTAL_SEEDS = 1000

PYSR_ROB_PATH = os.path.join(desktop_output, f'seed_resampling_pysr_{target_col}.xlsx')

#Load descriptors
X = dataset_descriptors_pruned[desc_cols_pruned].values
y = dataset_descriptors_pruned[target_col].values
groups = dataset_descriptors_pruned['metal_group'].values

#Resume from any seeds already recorded
if os.path.exists(PYSR_ROB_PATH):
    pysr_rob_df = pd.read_excel(PYSR_ROB_PATH)
    done_seeds = set(pysr_rob_df['seed'].tolist())
    pysr_rob_results = pysr_rob_df.to_dict('records')
else:
    done_seeds = set()
    pysr_rob_results = []

#Seeds in this batch that still need running
batch_todo = [s for s in PYSR_SEEDS if s not in done_seeds]

t_start = time.time()
for seed in PYSR_SEEDS:
    if seed in done_seeds:
        continue

    #Re-split at this seed
    gss_rob = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_idx, te_idx = next(gss_rob.split(X, y, groups=groups))
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    #Scale per seed and fit on train only 
    scaler_rob = StandardScaler()
    X_tr_s = scaler_rob.fit_transform(X_tr)
    X_te_s = scaler_rob.transform(X_te)

    #One PySR fit per seed
    try:
        m = PySRRegressor(**pysr_rob_kwargs)
        m.fit(X_tr_s, y_tr)
        yp_te = m.predict(X_te_s)
        yp_tr = m.predict(X_tr_s)
        best_eq_seed = m.get_best()
        pysr_rob_results.append({
            'seed': seed,
            'model': 'PySR',
            'train_rmse': round(root_mean_squared_error(y_tr, yp_tr), 4),
            'test_rmse': round(root_mean_squared_error(y_te, yp_te), 4),
            'train_r2': round(r2_score(y_tr, yp_tr), 4),
            'test_r2': round(r2_score(y_te, yp_te), 4),
            'train_mae': round(mean_absolute_error(y_tr, yp_tr), 4),
            'test_mae': round(mean_absolute_error(y_te, yp_te), 4),
            'complexity': int(best_eq_seed['complexity']),
            'loss': round(float(best_eq_seed['loss']), 6),
            'equation': str(best_eq_seed['equation']),
        })
    except Exception as e:
        print(f"Seed {seed}: FAILED — {e}")
        continue

    #Progress and incremental save
    n_done = len(pysr_rob_results) - len(done_seeds)
    if n_done % 10 == 0 and n_done > 0:
        pd.DataFrame(pysr_rob_results).to_excel(PYSR_ROB_PATH, index=False)
        elapsed = time.time() - t_start
        remaining = (len(batch_todo) - n_done) * elapsed / n_done
        print(f"  {len(done_seeds) + n_done}/{TOTAL_SEEDS} seeds | "
              f"{elapsed/3600:.2f}h elapsed | ~{remaining/3600:.2f}h left")

#Save seeds
pd.DataFrame(pysr_rob_results).to_excel(PYSR_ROB_PATH, index=False)

## 4.9. Model Comparison Under Resampling

In [ ]:
#Comparison of performance across all models when the train/test partition itself is varied

#Load seed resampling results and combine
rob_df = pd.read_excel(os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'))
pysr_df = pd.read_excel(os.path.join(desktop_output, f'seed_resampling_pysr_{target_col}.xlsx'))
rob_df = pd.concat([rob_df, pysr_df], ignore_index=True)

#Define performance of each model across seed resampling
resampling_stats = rob_df.groupby('model').agg(
    train_r2_mean=('train_r2', 'mean'), train_r2_std=('train_r2', 'std'),
    test_r2_mean=('test_r2', 'mean'), test_r2_std=('test_r2', 'std'),
    train_rmse_mean=('train_rmse', 'mean'), train_rmse_std=('train_rmse', 'std'),
    test_rmse_mean=('test_rmse', 'mean'), test_rmse_std=('test_rmse', 'std'),
    train_mae_mean=('train_mae', 'mean'), train_mae_std=('train_mae', 'std'),
    test_mae_mean=('test_mae', 'mean'), test_mae_std=('test_mae', 'std'),
    n_seeds=('test_rmse', 'count'),
).reset_index()

#Order by performance
resampling_stats = resampling_stats.sort_values('test_rmse_mean').reset_index(drop=True)

#Identify win-rate — the fraction of seeds each model has the lowest test RMSE
n_seeds_total = rob_df['seed'].nunique()
win_counts = rob_df.loc[rob_df.groupby('seed')['test_rmse'].idxmin(), 'model'].value_counts()
resampling_stats['win_rate'] = (resampling_stats['model'].map(win_counts).fillna(0)
                                / n_seeds_total * 100).round(1)

#Format per-model results
def ms(m, s, dp=3):
    return f"{m:.{dp}f} ± {s:.{dp}f}"
summary_table = pd.DataFrame({
    'Model': resampling_stats['model'],
    'Train R²': [ms(m, s) for m, s in zip(resampling_stats['train_r2_mean'],
                                          resampling_stats['train_r2_std'])],
    'Test R²': [ms(m, s) for m, s in zip(resampling_stats['test_r2_mean'],
                                         resampling_stats['test_r2_std'])],
    'Train RMSE': [ms(m, s) for m, s in zip(resampling_stats['train_rmse_mean'],
                                            resampling_stats['train_rmse_std'])],
    'Test RMSE': [ms(m, s) for m, s in zip(resampling_stats['test_rmse_mean'],
                                           resampling_stats['test_rmse_std'])],
    'Train MAE': [ms(m, s) for m, s in zip(resampling_stats['train_mae_mean'],
                                           resampling_stats['train_mae_std'])],
    'Test MAE': [ms(m, s) for m, s in zip(resampling_stats['test_mae_mean'],
                                          resampling_stats['test_mae_std'])],
    'Win Rate': [f"{w:.1f}%" for w in resampling_stats['win_rate']],
})

#Export results
with pd.ExcelWriter(os.path.join(
        desktop_output, f'model_comparison_resampling_{target_col}.xlsx')) as writer:
    summary_table.to_excel(writer, sheet_name='Formatted', index=False)
    resampling_stats.round(4).to_excel(writer, sheet_name='Numeric', index=False)

# 5. Oxygen Evolution Reaction

## 5.1. Descriptor Selection

### 5.1.1. Train/Test Split

In [ ]:
#Train/test split in an 80/20 ratio
    #Train/test split performed before descriptor selection to prevent data leakage
    #Random seed set to ensure reproducibility

#Reload the full descriptor dataset 
dataset_descriptors = pd.read_csv(os.path.join(base_dir, 'dataset_descriptors.csv'))

#Define target and file path
    #Target is delta_G_H for the HER and eta_OER for the OER
target_col = 'eta_OER'
reaction = 'HER' if target_col == 'delta_G_H' else 'OER'
units = 'eV' if target_col == 'delta_G_H' else 'V'
desktop_output = os.path.join(base_dir, reaction)
os.makedirs(desktop_output, exist_ok=True)

#Define train/test split ratio and set random seed
test_size = 0.20
random_state = 42
gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
train_idx, test_idx = next(gss.split(
    dataset_descriptors,
    dataset_descriptors[target_col],
    groups=dataset_descriptors['metal_group']
))

#Partition dataset into train and test dataframes
    #Test set is stored away and not used again until machine learning model evaluation
train_df = dataset_descriptors.iloc[train_idx].copy().reset_index(drop=True)
test_df = dataset_descriptors.iloc[test_idx].copy().reset_index(drop=True)

#Verify no group leakage between train and test sets
group_leakage = len(set(train_df['metal_group']) & set(test_df['metal_group']))
assert group_leakage == 0, "Warning: Group leakage detected"

### 5.1.2. Descriptor Pruning

#### 5.1.2.1. Mutual Information Analysis on Full Descriptor Set

In [ ]:
#Mutual Information analysis
    #Computed on full descriptor set and training data only

#Define full descriptor columns and training data
X_train_desc = train_df[desc_cols].copy()
y_train_mi = train_df[target_col].copy()

#Compute Mutual Information scores against the target on training data only
scores = mutual_info_regression(
    X_train_desc.values,
    y_train_mi.values,
    random_state=42
)
mi_df = pd.Series(scores, index=desc_cols).sort_values(ascending=False)
mi_df = mi_df.to_frame(name='MI_score')

#Export Mutual Information results
mi_df.round(4).to_excel(os.path.join(desktop_output, f'mutual_information_{target_col}.xlsx'))

#### 5.1.2.2. Pearson Correlation Analysis on Full Descriptor Set

In [ ]:
#Pearson Correlation analysis
    #Identify perfectly and highly correlated descriptor pairs (r >= 0.98)
    #For each such pair, the descriptor with the lower Mutual Information score will be pruned
    #Computed on training data only

X_train_corr = train_df[desc_cols].copy()

#Compute Pearson correlation matrix
pearson_corr = X_train_corr.corr(method='pearson')

#Export all pairs
all_pairs = []
cols = pearson_corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = pearson_corr.iloc[i, j]
        all_pairs.append({
            'Feature_1': cols[i],
            'Feature_2': cols[j],
            'Correlation': round(r, 4),
        })
pearson_all_pairs = pd.DataFrame(all_pairs).sort_values('Correlation', key=abs, ascending=False)
pearson_all_pairs.to_excel(os.path.join(desktop_output, f'pearson_correlation_{target_col}.xlsx'), index=False)

#Identify perfectly and highly correlated pairs (r >= 0.98)
CORR_THRESH = 0.98
pearson_high = [(cols[i], cols[j])
                for i in range(len(cols)) for j in range(i+1, len(cols))
                if abs(pearson_corr.iloc[i, j]) >= CORR_THRESH]

#### 5.1.2.3. Spearman Correlation Analysis on Full Descriptor Set

In [ ]:
#Spearman Correlation analysis
    #Identify perfectly and highly correlated descriptor pairs (r >= 0.98)
    #For each such pair, the descriptor with the lower Mutual Information score will be pruned
    #Computed on training data only

#Compute Spearman correlation matrix
spearman_corr = X_train_corr.corr(method='spearman')

#Export all pairs
all_pairs = []
cols = spearman_corr.columns
for i in range(len(cols)):
    for j in range(i+1, len(cols)):
        r = spearman_corr.iloc[i, j]
        all_pairs.append({
            'Feature_1': cols[i],
            'Feature_2': cols[j],
            'Correlation': round(r, 4),
        })
spearman_all_pairs = pd.DataFrame(all_pairs).sort_values('Correlation', key=abs, ascending=False)
spearman_all_pairs.to_excel(os.path.join(desktop_output, f'spearman_correlation_{target_col}.xlsx'), index=False)

#Identify perfectly and highly correlated pairs (p >= 0.98)
spearman_high = [(cols[i], cols[j])
                 for i in range(len(cols)) for j in range(i+1, len(cols))
                 if abs(spearman_corr.iloc[i, j]) >= CORR_THRESH]

#Combine correlated pairs from both Pearson and Spearman and deduplicate by the lower Mutual Information score
high_pairs = list(set(pearson_high) | set(spearman_high))
corr_drops = []
for a, b in high_pairs:

    #Retain the higher Mutual Information descriptor and drop the lower Mutual Information descriptor of the pair
    mi_a = mi_df.loc[a, 'MI_score']
    mi_b = mi_df.loc[b, 'MI_score']
    drop = b if mi_a >= mi_b else a
    if drop not in corr_drops:
        corr_drops.append(drop)

#Apply deduplication to form the descriptor set passed to Boruta
desc_cols_dedup = [c for c in desc_cols if c not in corr_drops]

#### 5.1.2.4. Boruta on Pearson and Spearman Correlation Pruned Descriptor Set

In [ ]:
#Run Boruta on descriptor set
boruta = BorutaPy(
    RandomForestRegressor(n_jobs=-1, random_state=42, max_depth=5),
    n_estimators='auto',
    max_iter=1000,
    random_state=42,
    verbose=1
)
boruta.fit(
    train_df[desc_cols_dedup].values,
    train_df[target_col].values
)

#Identify confirmed, tentative, and rejected descriptor sets
confirmed_mask = boruta.support_
tentative_mask = boruta.support_weak_
desc_cols_confirmed = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup)) if confirmed_mask[i]]
desc_cols_tentative = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup)) if tentative_mask[i]]
desc_cols_rejected = [desc_cols_dedup[i] for i in range(len(desc_cols_dedup))
                      if not confirmed_mask[i] and not tentative_mask[i]]

#Use confirmed descriptors only as working descriptor set
desc_cols_pruned = desc_cols_confirmed.copy()

#Export Boruta results
boruta_df = pd.DataFrame({
    'descriptor': desc_cols_dedup,
    'status': ['Confirmed' if confirmed_mask[i]
               else 'Tentative' if tentative_mask[i]
               else 'Rejected'
               for i in range(len(desc_cols_dedup))],
    'ranking': boruta.ranking_
}).sort_values('ranking')
boruta_df.to_excel(os.path.join(desktop_output, f'boruta_{target_col}.xlsx'), index=False)

#### 5.1.2.5. Final Descriptor Set

In [ ]:
#Export final descriptor dataset
dataset_descriptors_pruned = dataset_descriptors[
    ['metal_A', 'metal_B', 'metal_group',
     'energy_slab', 'energy_slab_adsorbate_H', 'energy_slab_adsorbate_O',
     'energy_slab_adsorbate_OH', 'energy_slab_adsorbate_OOH',
     'delta_E_H', 'delta_E_O', 'delta_E_OH', 'delta_E_OOH',
     'delta_G_H', 'delta_G_OH', 'delta_G_O', 'delta_G_OOH',
     'delta_G1', 'delta_G2', 'delta_G3', 'delta_G4', 'eta_OER'] + desc_cols_pruned]
dataset_descriptors_pruned.to_csv(
    os.path.join(desktop_output, f'dataset_descriptors_pruned_{target_col}.csv'), index=False)

#### 5.1.2.6. Pearson and Spearman Correlation and Mutual Information Analysis on Final Descriptor Set

In [ ]:
#Pearson and Spearman Correlation and Mutual Information analysis on final descriptor set
    #Computed on final descriptor set and training data only

X_train_final = train_df[desc_cols_pruned]

#Pearson and Spearman correlation of every descriptor pair
for method in ['pearson', 'spearman']:
    corr = X_train_final.corr(method=method)
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
    pairs = (upper.rename('correlation').rename_axis(['descriptor_1', 'descriptor_2']).reset_index()
             .assign(abs_correlation=lambda d: d['correlation'].abs())
             .sort_values('abs_correlation', ascending=False))
    pairs.to_excel(os.path.join(desktop_output, f'{method}_correlation_{target_col}_final.xlsx'), index=False)

#Mutual Information scores against the target on training data only
scores = mutual_info_regression(
    X_train_final.values,
    train_df[target_col].values,
    random_state=42
)
mi_df_final = pd.Series(scores, index=desc_cols_pruned).sort_values(ascending=False)
mi_df_final = mi_df_final.to_frame(name='MI_score')
mi_df_final.round(4).to_excel(os.path.join(desktop_output, f'mutual_information_{target_col}_final.xlsx'))

## 5.2. Data Pre-Processing

In [ ]:
#Reload dataset
dataset_descriptors_pruned = pd.read_csv(os.path.join(desktop_output, f'dataset_descriptors_pruned_{target_col}.csv'))
train_df = dataset_descriptors_pruned.iloc[train_idx].copy().reset_index(drop=True)
test_df = dataset_descriptors_pruned.iloc[test_idx].copy().reset_index(drop=True)

#Define X (descriptors) and y (target)
X_train = train_df[desc_cols_pruned].values
X_test = test_df[desc_cols_pruned].values
y_train = train_df[target_col].values
y_test = test_df[target_col].values
groups_train = train_df['metal_group'].values
groups_test = test_df['metal_group'].values

#Descriptor scaling
    #Scaler fit on training data only, then applied to test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#Establish SHAP configuration, which is applied uniformly to all models via KernelExplainer (excluding PySR)
    #Background = 50 k-means centroids of the training set 
    #nsamples = 200 coalitions per explanation
    #l1_reg = 0 so that importance is estimated for all descriptors
np.random.seed(42)
shap_background = shap.kmeans(X_train_scaled, 50)
shap_nsamples = 200
shap_l1_reg = 0

#Initialise results storage
results = []
all_predictions = pd.DataFrame()
shap_results = pd.DataFrame()
hyperparam_configs = pd.DataFrame()

## 5.3. Baseline Model

### 5.3.1. Mean Predictor

In [ ]:
#Mean Predictor
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html

#Define cross-validation strategy
outer_cv = GroupKFold(n_splits=5)

#Fit Mean Predictor
mean_pred = DummyRegressor(strategy='mean')
mean_pred.fit(X_train_scaled, y_train)

#Cross-validation using GroupKFold(5)
cv_scores_mean = cross_val_score(
    DummyRegressor(strategy='mean'), X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_mean = -cv_scores_mean.mean()
cv_rmse_std_mean = cv_scores_mean.std()

#Mean Predictor predictions
y_pred_train_mean = mean_pred.predict(X_train_scaled)
y_pred_test_mean = mean_pred.predict(X_test_scaled)

#Mean Predictor performance metrics
train_r2_mean = r2_score(y_train, y_pred_train_mean)
test_r2_mean = r2_score(y_test, y_pred_test_mean)
train_rmse_mean = root_mean_squared_error(y_train, y_pred_train_mean)
test_rmse_mean = root_mean_squared_error(y_test, y_pred_test_mean)
train_mae_mean = mean_absolute_error(y_train, y_pred_train_mean)
test_mae_mean = mean_absolute_error(y_test, y_pred_test_mean)

#Store Mean Predictor results
results.append({
    'model': 'MP',
    'cv_rmse': round(cv_rmse_mean, 4),
    'cv_rmse_std': round(cv_rmse_std_mean, 4),
    'train_r2': round(train_r2_mean, 4),
    'test_r2': round(test_r2_mean, 4),
    'train_rmse': round(train_rmse_mean, 4),
    'test_rmse': round(test_rmse_mean, 4),
    'train_mae': round(train_mae_mean, 4),
    'test_mae': round(test_mae_mean, 4),
    'best_params': 'strategy=mean'
})

#Store Mean Predictor predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'MP',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_mean
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'MP',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_mean
})], ignore_index=True)

#Record Mean Predictor results for Rashomon set
cv_df_mp = pd.DataFrame([{'model': 'MP', 'cv_rmse': cv_rmse_mean,
                          'cv_rmse_std': cv_rmse_std_mean}])
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_mp], ignore_index=True)

## 5.4. Machine Learning Models

### 5.4.1. Linear Regression and SHapley Additive exPlanations

In [ ]:
#Linear Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html

#Define cross-validation strategy
outer_cv = GroupKFold(n_splits=5)

#Fit Linear Regression on full training set
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

#Cross-validation using GroupKFold(5)
cv_scores_lr = cross_val_score(
    lr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_lr = -cv_scores_lr.mean()
cv_rmse_std_lr = cv_scores_lr.std()

#Linear Regression predictions
y_pred_train_lr = lr.predict(X_train_scaled)
y_pred_test_lr = lr.predict(X_test_scaled)

#Linear Regression performance metrics
train_r2_lr = r2_score(y_train, y_pred_train_lr)
test_r2_lr = r2_score(y_test, y_pred_test_lr)
train_rmse_lr = root_mean_squared_error(y_train, y_pred_train_lr)
test_rmse_lr = root_mean_squared_error(y_test, y_pred_test_lr)
train_mae_lr = mean_absolute_error(y_train, y_pred_train_lr)
test_mae_lr = mean_absolute_error(y_test, y_pred_test_lr)

#Store Linear Regression hyperparameters
lr_params = {
    'fit_intercept': lr.fit_intercept,
    'positive': lr.positive,
}

#Store Linear Regression results
results.append({
    'model': 'LR',
    'cv_rmse': round(cv_rmse_lr, 4),
    'cv_rmse_std': round(cv_rmse_std_lr, 4),
    'train_r2': round(train_r2_lr, 4),
    'test_r2': round(test_r2_lr, 4),
    'train_rmse': round(train_rmse_lr, 4),
    'test_rmse': round(test_rmse_lr, 4),
    'train_mae': round(train_mae_lr, 4),
    'test_mae': round(test_mae_lr, 4),
    'best_params': str(lr_params)
})

#Store Linear Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'LR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_lr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'LR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_lr
})], ignore_index=True)

#SHAP on Linear Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_lr = shap.KernelExplainer(lr.predict, shap_background)
shap_values_lr = explainer_lr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Linear Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'LR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_lr).mean(axis=0)
})], ignore_index=True)

#Record Linear Regression results for Rashomon set
cv_df_lr = pd.DataFrame([{'model': 'LR', 'cv_rmse': cv_rmse_lr,
                          'cv_rmse_std': cv_rmse_std_lr}])
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_lr], ignore_index=True)

### 5.4.2. Random Forest Regression and SHapley Additive exPlanations

In [ ]:
#Random Forest Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html

#Default parameters:    
    #n_estimators = 100    
    #max_depth = None    
    #min_samples_split = 2    
    #min_samples_leaf = 1    
    #max_features = 1.0

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_rfr = {
    'n_estimators': randint(100, 1500),
    'max_depth': [None, 4, 6, 8, 10, 12, 15, 20, 30, 40],
    'min_samples_split': randint(2, 21),
    'min_samples_leaf': randint(1, 11),
    'max_features': ['sqrt', 'log2', 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, None],
    'bootstrap': [True, False],
    'criterion': ['squared_error', 'absolute_error'],
    'min_weight_fraction_leaf': [0.0, 0.01, 0.02, 0.05],
    'max_leaf_nodes': [None, 20, 50, 100, 200],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_rfr = 1000
search_rfr = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_dist_rfr,
    n_iter=n_iter_rfr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)

search_rfr.fit(X_train_scaled, y_train, groups=groups_train)
rfr = search_rfr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_rfr = cross_val_score(
    rfr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_rfr = -cv_scores_rfr.mean()
cv_rmse_std_rfr = cv_scores_rfr.std()

#Random Forest Regression predictions
y_pred_train_rfr = rfr.predict(X_train_scaled)
y_pred_test_rfr = rfr.predict(X_test_scaled)

#Random Forest Regression performance metrics
train_r2_rfr = r2_score(y_train, y_pred_train_rfr)
test_r2_rfr = r2_score(y_test, y_pred_test_rfr)
train_rmse_rfr = root_mean_squared_error(y_train, y_pred_train_rfr)
test_rmse_rfr = root_mean_squared_error(y_test, y_pred_test_rfr)
train_mae_rfr = mean_absolute_error(y_train, y_pred_train_rfr)
test_mae_rfr = mean_absolute_error(y_test, y_pred_test_rfr)

#Store best Random Forest Regression hyperparameters
rfr_params = {
    'n_estimators': rfr.n_estimators,
    'max_depth': rfr.max_depth,
    'min_samples_split': rfr.min_samples_split,
    'min_samples_leaf': rfr.min_samples_leaf,
    'max_features': rfr.max_features,
    'bootstrap': rfr.bootstrap,
    'criterion': rfr.criterion,
    'min_weight_fraction_leaf': rfr.min_weight_fraction_leaf,
    'max_leaf_nodes': rfr.max_leaf_nodes,
}

#Store Random Forest Regression results
results.append({
    'model': 'RFR',
    'cv_rmse': round(cv_rmse_rfr, 4),
    'cv_rmse_std': round(cv_rmse_std_rfr, 4),
    'train_r2': round(train_r2_rfr, 4),
    'test_r2': round(test_r2_rfr, 4),
    'train_rmse': round(train_rmse_rfr, 4),
    'test_rmse': round(test_rmse_rfr, 4),
    'train_mae': round(train_mae_rfr, 4),
    'test_mae': round(test_mae_rfr, 4),
    'best_params': str(rfr_params)
})

#Store Random Forest Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'RFR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_rfr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'RFR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_rfr
})], ignore_index=True)

#SHAP on Random Forest Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_rfr = shap.KernelExplainer(rfr.predict, shap_background)
shap_values_rfr = explainer_rfr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Random Forest Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'RFR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_rfr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Random Forest Regression hyperparameter configurations for Rashomon set
cv_df_rfr = pd.DataFrame(search_rfr.cv_results_['params'])
cv_df_rfr.insert(0, 'model', 'RFR')
cv_df_rfr['cv_rmse'] = -search_rfr.cv_results_['mean_test_score']
cv_df_rfr['cv_rmse_std'] = search_rfr.cv_results_['std_test_score']
cv_df_rfr = cv_df_rfr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_rfr], ignore_index=True)

### 5.4.3. Gradient Boosting Regression and SHapley Additive exPlanations

In [ ]:
#Gradient Boosting Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html

#Default parameters:
    #learning_rate = 0.1
    #n_estimators = 100
    #max_depth = 3
    #subsample = 1.0
    #max_features = None
    #min_samples_split = 2
    #min_samples_leaf = 1

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_gbr = {
    'learning_rate': uniform(0.01, 0.29),         
    'n_estimators': randint(100, 1000),
    'max_depth': [2, 3, 4, 5, 6],
    'subsample': uniform(0.5, 0.5),               
    'max_features': ['sqrt', 'log2', 0.3, 0.4, 0.5, 0.6, 0.7, None],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 5),
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested 
n_iter_gbr = 1000
search_gbr = RandomizedSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_distributions=param_dist_gbr,
    n_iter=n_iter_gbr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_gbr.fit(X_train_scaled, y_train, groups=groups_train)
gbr = search_gbr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_gbr = cross_val_score(
    gbr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_gbr = -cv_scores_gbr.mean()
cv_rmse_std_gbr = cv_scores_gbr.std()

#Gradient Boosting Regression predictions
y_pred_train_gbr = gbr.predict(X_train_scaled)
y_pred_test_gbr = gbr.predict(X_test_scaled)

#Gradient Boosting Regression performance metrics
train_r2_gbr = r2_score(y_train, y_pred_train_gbr)
test_r2_gbr = r2_score(y_test, y_pred_test_gbr)
train_rmse_gbr = root_mean_squared_error(y_train, y_pred_train_gbr)
test_rmse_gbr = root_mean_squared_error(y_test, y_pred_test_gbr)
train_mae_gbr = mean_absolute_error(y_train, y_pred_train_gbr)
test_mae_gbr = mean_absolute_error(y_test, y_pred_test_gbr)

#Store best Gradient Boosting Regression hyperparameters
gbr_params = {
    'learning_rate': gbr.learning_rate,
    'n_estimators': gbr.n_estimators,
    'max_depth': gbr.max_depth,
    'subsample': gbr.subsample,
    'max_features': gbr.max_features,
    'min_samples_split': gbr.min_samples_split,
    'min_samples_leaf': gbr.min_samples_leaf,
    'criterion': gbr.criterion,
    'loss': gbr.loss,
}

#Store Gradient Boosting Regression results
results.append({
    'model': 'GBR',
    'cv_rmse': round(cv_rmse_gbr, 4),
    'cv_rmse_std': round(cv_rmse_std_gbr, 4),
    'train_r2': round(train_r2_gbr, 4),
    'test_r2': round(test_r2_gbr, 4),
    'train_rmse': round(train_rmse_gbr, 4),
    'test_rmse': round(test_rmse_gbr, 4),
    'train_mae': round(train_mae_gbr, 4),
    'test_mae': round(test_mae_gbr, 4),
    'best_params': str(gbr_params)
})

#Store Gradient Boosting Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'GBR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_gbr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'GBR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_gbr
})], ignore_index=True)

#SHAP on Gradient Boosting Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_gbr = shap.KernelExplainer(gbr.predict, shap_background)
shap_values_gbr = explainer_gbr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Gradient Boosting Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'GBR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_gbr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Gradient Boosting Regression hyperparameter configurations for Rashomon set
cv_df_gbr = pd.DataFrame(search_gbr.cv_results_['params'])
cv_df_gbr.insert(0, 'model', 'GBR')
cv_df_gbr['cv_rmse'] = -search_gbr.cv_results_['mean_test_score']
cv_df_gbr['cv_rmse_std'] = search_gbr.cv_results_['std_test_score']
cv_df_gbr = cv_df_gbr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_gbr], ignore_index=True)

### 5.4.4. Support Vector Regression and SHapley Additive exPlanations

In [ ]:
#Support Vector Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html

#Default parameters:
    #C = 1.0
    #epsilon = 0.1
    #kernel = 'rbf'
    #gamma = 'scale'

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_svr = {
    'kernel': ['rbf', 'poly'],
    'C': loguniform(0.1, 1000),
    'epsilon': uniform(0.01, 0.29),
    'gamma': loguniform(1e-4, 1.0),
    'degree': randint(2, 5), #poly only (not relevant for rbf)
    'coef0': uniform(0.0, 1.0), #poly only (not relevant for rbf)
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_svr = 1000
search_svr = RandomizedSearchCV(
    SVR(),
    param_distributions=param_dist_svr,
    n_iter=n_iter_svr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_svr.fit(X_train_scaled, y_train, groups=groups_train)
svr = search_svr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_svr = cross_val_score(
    svr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_svr = -cv_scores_svr.mean()
cv_rmse_std_svr = cv_scores_svr.std()

#Support Vector Regression predictions
y_pred_train_svr = svr.predict(X_train_scaled)
y_pred_test_svr = svr.predict(X_test_scaled)

#Support Vector Regression performance metrics
train_r2_svr = r2_score(y_train, y_pred_train_svr)
test_r2_svr = r2_score(y_test, y_pred_test_svr)
train_rmse_svr = root_mean_squared_error(y_train, y_pred_train_svr)
test_rmse_svr = root_mean_squared_error(y_test, y_pred_test_svr)
train_mae_svr = mean_absolute_error(y_train, y_pred_train_svr)
test_mae_svr = mean_absolute_error(y_test, y_pred_test_svr)

#Store best Support Vector Regression hyperparameters
svr_params = {
    'kernel': svr.kernel,
    'C': svr.C,
    'epsilon': svr.epsilon,
    'gamma': svr.gamma,
    'degree': svr.degree,
    'coef0': svr.coef0,
}

#Store Support Vector Regression results
results.append({
    'model': 'SVR',
    'cv_rmse': round(cv_rmse_svr, 4),
    'cv_rmse_std': round(cv_rmse_std_svr, 4),
    'train_r2': round(train_r2_svr, 4),
    'test_r2': round(test_r2_svr, 4),
    'train_rmse': round(train_rmse_svr, 4),
    'test_rmse': round(test_rmse_svr, 4),
    'train_mae': round(train_mae_svr, 4),
    'test_mae': round(test_mae_svr, 4),
    'best_params': str(svr_params)
})

#Store Support Vector Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'SVR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_svr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'SVR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_svr
})], ignore_index=True)

#SHAP on Support Vector Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_svr = shap.KernelExplainer(svr.predict, shap_background)
shap_values_svr = explainer_svr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Support Vector Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'SVR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_svr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Support Vector Regression hyperparameter configurations for Rashomon set
cv_df_svr = pd.DataFrame(search_svr.cv_results_['params'])
cv_df_svr.insert(0, 'model', 'SVR')
cv_df_svr['cv_rmse'] = -search_svr.cv_results_['mean_test_score']
cv_df_svr['cv_rmse_std'] = search_svr.cv_results_['std_test_score']
cv_df_svr = cv_df_svr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_svr], ignore_index=True)

### 5.4.5. Gaussian Process Regression and SHapley Additive exPlanations

In [ ]:
#Gaussian Process Regression
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.gaussian_process.GaussianProcessRegressor.html

#Default parameters:
    #kernel = None 
    #alpha = 1e-10
    #n_restarts_optimizer = 0
    #normalize_y = False

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Candidate kernel structures
kernels_gpr = [
    C(1.0) * RBF(1.0) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=0.5) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=1.5) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=2.5) + WhiteKernel(0.1),
    C(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(0.1),
    C(1.0) * RBF(1.0) + C(1.0) * RationalQuadratic(1.0, 1.0) + WhiteKernel(0.1),
    C(1.0) * Matern(nu=1.5) + C(1.0) * RationalQuadratic(1.0, 1.0) + WhiteKernel(0.1),
    C(1.0) * RBF(length_scale=[1.0] * len(desc_cols_pruned)) + WhiteKernel(0.1),
]

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_gpr = {
    'kernel': kernels_gpr,
    'alpha': loguniform(1e-10, 1e-1),
    'n_restarts_optimizer': [0, 1, 2, 3],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested
n_iter_gpr = 1000
search_gpr = RandomizedSearchCV(
    GaussianProcessRegressor(normalize_y=True, random_state=42),
    param_distributions=param_dist_gpr,
    n_iter=n_iter_gpr,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_gpr.fit(X_train_scaled, y_train, groups=groups_train)
gpr = search_gpr.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_gpr = cross_val_score(
    gpr, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_gpr = -cv_scores_gpr.mean()
cv_rmse_std_gpr = cv_scores_gpr.std()

#Gaussian Process Regression predictions
y_pred_train_gpr = gpr.predict(X_train_scaled)
y_pred_test_gpr = gpr.predict(X_test_scaled)

#Gaussian Process Regression performance metrics
train_r2_gpr = r2_score(y_train, y_pred_train_gpr)
test_r2_gpr = r2_score(y_test, y_pred_test_gpr)
train_rmse_gpr = root_mean_squared_error(y_train, y_pred_train_gpr)
test_rmse_gpr = root_mean_squared_error(y_test, y_pred_test_gpr)
train_mae_gpr = mean_absolute_error(y_train, y_pred_train_gpr)
test_mae_gpr = mean_absolute_error(y_test, y_pred_test_gpr)

#Store best Gaussian Process Regression hyperparameters
gpr_params = {
    'kernel': str(gpr.kernel_),
    'alpha': gpr.alpha,
    'n_restarts_optimizer': gpr.n_restarts_optimizer,
    'normalize_y': gpr.normalize_y,
}

#Store Gaussian Process Regression results
results.append({
    'model': 'GPR',
    'cv_rmse': round(cv_rmse_gpr, 4),
    'cv_rmse_std': round(cv_rmse_std_gpr, 4),
    'train_r2': round(train_r2_gpr, 4),
    'test_r2': round(test_r2_gpr, 4),
    'train_rmse': round(train_rmse_gpr, 4),
    'test_rmse': round(test_rmse_gpr, 4),
    'train_mae': round(train_mae_gpr, 4),
    'test_mae': round(test_mae_gpr, 4),
    'best_params': str(gpr_params)
})

#Store Gaussian Process Regression predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'GPR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_gpr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'GPR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_gpr
})], ignore_index=True)

#SHAP on Gaussian Process Regression — KernelExplainer on the test set
np.random.seed(42)
explainer_gpr = shap.KernelExplainer(gpr.predict, shap_background)
shap_values_gpr = explainer_gpr.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Gaussian Process Regression SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'GPR',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_gpr).mean(axis=0)
})], ignore_index=True)

#Record all sampled Gaussian Process Regression hyperparameter configurations for Rashomon set
cv_df_gpr = pd.DataFrame(search_gpr.cv_results_['params'])
cv_df_gpr['kernel'] = cv_df_gpr['kernel'].astype(str)
cv_df_gpr.insert(0, 'model', 'GPR')
cv_df_gpr['cv_rmse'] = -search_gpr.cv_results_['mean_test_score']
cv_df_gpr['cv_rmse_std'] = search_gpr.cv_results_['std_test_score']
cv_df_gpr = cv_df_gpr.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_gpr], ignore_index=True)

### 5.4.6. Multilayer Perceptron and SHapley Additive exPlanations

In [ ]:
#Multilayer Perceptron
    #Implementation: https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html

#Default parameters:
    #hidden_layer_sizes = (100,)
    #activation = 'relu'
    #solver = 'adam'
    #alpha = 0.0001
    #learning_rate_init = 0.001
    #max_iter = 200

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#Hyperparameter search space sampled by RandomizedSearchCV
param_dist_mlp = {
    'hidden_layer_sizes': [(32,), (64,), (128,),
                           (32, 16), (64, 32), (128, 64), (64, 32, 16)],
    'activation': ['relu', 'tanh'],
    'alpha': loguniform(1e-5, 3),        
    'learning_rate_init': loguniform(1e-4, 1e-1),
    'solver': ['adam', 'lbfgs'],
}

#Hyperparameter tuning via RandomizedSearchCV with inner GroupKFold(3)
    #Number of hyperparameter configurations tested 
n_iter_mlp = 1000
search_mlp = RandomizedSearchCV(
    MLPRegressor(random_state=42, max_iter=2000, early_stopping=False),
    param_distributions=param_dist_mlp,
    n_iter=n_iter_mlp,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    refit=True,
    random_state=42,
    verbose=1
)
search_mlp.fit(X_train_scaled, y_train, groups=groups_train)
mlp = search_mlp.best_estimator_

#Cross-validation using GroupKFold(5) on tuned model
cv_scores_mlp = cross_val_score(
    mlp, X_train_scaled, y_train,
    cv=outer_cv,
    groups=groups_train,
    scoring='neg_root_mean_squared_error'
)
cv_rmse_mlp = -cv_scores_mlp.mean()
cv_rmse_std_mlp = cv_scores_mlp.std()

#Multilayer Perceptron predictions
y_pred_train_mlp = mlp.predict(X_train_scaled)
y_pred_test_mlp = mlp.predict(X_test_scaled)

#Multilayer Perceptron performance metrics
train_r2_mlp = r2_score(y_train, y_pred_train_mlp)
test_r2_mlp = r2_score(y_test, y_pred_test_mlp)
train_rmse_mlp = root_mean_squared_error(y_train, y_pred_train_mlp)
test_rmse_mlp = root_mean_squared_error(y_test, y_pred_test_mlp)
train_mae_mlp = mean_absolute_error(y_train, y_pred_train_mlp)
test_mae_mlp = mean_absolute_error(y_test, y_pred_test_mlp)

#Store best Multilayer Perceptron hyperparameters
mlp_params = {
    'hidden_layer_sizes': mlp.hidden_layer_sizes,
    'activation': mlp.activation,
    'solver': mlp.solver,
    'alpha': mlp.alpha,
    'learning_rate_init': mlp.learning_rate_init,
    'max_iter': mlp.max_iter,
}

#Store Multilayer Perceptron results
results.append({
    'model': 'MLP',
    'cv_rmse': round(cv_rmse_mlp, 4),
    'cv_rmse_std': round(cv_rmse_std_mlp, 4),
    'train_r2': round(train_r2_mlp, 4),
    'test_r2': round(test_r2_mlp, 4),
    'train_rmse': round(train_rmse_mlp, 4),
    'test_rmse': round(test_rmse_mlp, 4),
    'train_mae': round(train_mae_mlp, 4),
    'test_mae': round(test_mae_mlp, 4),
    'best_params': str(mlp_params)
})

#Store Multilayer Perceptron predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'MLP',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_mlp
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'MLP',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_mlp
})], ignore_index=True)

#SHAP on Multilayer Perceptron — KernelExplainer on the test set
np.random.seed(42)
explainer_mlp = shap.KernelExplainer(mlp.predict, shap_background)
shap_values_mlp = explainer_mlp.shap_values(X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)

#Store Multilayer Perceptron SHAP results
shap_results = pd.concat([shap_results, pd.DataFrame({
    'model': 'MLP',
    'descriptor': desc_cols_pruned,
    'mean_abs_shap': np.abs(shap_values_mlp).mean(axis=0)
})], ignore_index=True)

#Record all sampled Multilayer Perceptron hyperparameter configurations for Rashomon set
cv_df_mlp = pd.DataFrame(search_mlp.cv_results_['params'])
cv_df_mlp.insert(0, 'model', 'MLP')
cv_df_mlp['cv_rmse'] = -search_mlp.cv_results_['mean_test_score']
cv_df_mlp['cv_rmse_std'] = search_mlp.cv_results_['std_test_score']
cv_df_mlp = cv_df_mlp.sort_values('cv_rmse').reset_index(drop=True)
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_mlp], ignore_index=True)

### 5.4.7. Python Symbolic Regression

In [ ]:
#PySR
    #Implementation: https://github.com/MilesCranmer/PySR

#Define cross-validation strategy
inner_cv = GroupKFold(n_splits=3)
outer_cv = GroupKFold(n_splits=5)

#PySR configuration
pysr_base = dict(
    niterations=200,
    binary_operators=["+", "-", "*"],
    unary_operators=["square", "cube", "abs"],
    populations=30,
    population_size=50,
    constraints={"square": 9, "cube": 6},
    nested_constraints={"square": {"square": 0, "cube": 0},
                        "cube":   {"square": 0, "cube": 0}},
    model_selection="best",
    random_state=42,
    progress=False,
    verbosity=0,
)

#Deterministic settings for fits, and parallel settings for tuning search
pysr_deterministic = dict(deterministic=True, procs=0, multithreading=False)
pysr_parallel = dict(deterministic=False, parallelism="multithreading", procs=4)

#Hyperparameter tuning
    #Tuned on the training data only using inner GroupKFold(3)
param_grid_pysr = {
    'maxsize': [15, 20, 25, 30, 35, 40, 50, 60],
    'parsimony': [1e-5, 1e-4, 1e-3, 1e-2],
}

inner_folds = list(inner_cv.split(X_train_scaled, y_train, groups=groups_train))
pysr_cv_records = []

for ms, pars in itertools.product(param_grid_pysr['maxsize'], param_grid_pysr['parsimony']):
    t_cfg = time.time()
    fold_rmse, fold_cplx = [], []
    for tr_i, va_i in inner_folds:
        model_i = PySRRegressor(**{**pysr_base, **pysr_parallel,
                                   'maxsize': ms, 'parsimony': pars})
        model_i.fit(X_train_scaled[tr_i], y_train[tr_i])
        pred_i = model_i.predict(X_train_scaled[va_i])
        n_bad = int((~np.isfinite(pred_i)).sum())
        if n_bad:
            print(f"Warning: {n_bad} non-finite predictions")
            pred_i = np.where(np.isfinite(pred_i), pred_i, y_train[tr_i].mean())
        fold_rmse.append(root_mean_squared_error(y_train[va_i], pred_i))
        fold_cplx.append(int(model_i.get_best()['complexity']))
    pysr_cv_records.append({
        'maxsize': ms,
        'parsimony': pars,
        'cv_rmse': np.mean(fold_rmse),
        'cv_rmse_sd': np.std(fold_rmse),
        'mean_complexity': np.mean(fold_cplx),
        'minutes': (time.time() - t_cfg) / 60,
    })
    r = pysr_cv_records[-1]
    print(f"  maxsize={ms:<3} parsimony={pars:<7} | inner CV RMSE {r['cv_rmse']:.4f} "
          f"± {r['cv_rmse_sd']:.4f} | complexity {r['mean_complexity']:.1f} "
          f"| {r['minutes']:.1f} min")

cv_df_pysr = pd.DataFrame(pysr_cv_records).sort_values('cv_rmse').reset_index(drop=True)

#Select the configuration with the lowest inner CV RMSE
best_cfg = cv_df_pysr.iloc[0]
pysr_maxsize = int(best_cfg['maxsize'])
pysr_parsimony = float(best_cfg['parsimony'])

#Assemble the tuned configuration used for all fits
pysr_kwargs = {**pysr_base, **pysr_deterministic,
               'maxsize': pysr_maxsize, 'parsimony': pysr_parsimony}

#Cross-validation using GroupKFold(5) on the tuned configuration
cv_rmse_folds = []
for fold_i, (tr, va) in enumerate(outer_cv.split(X_train_scaled, y_train, groups=groups_train)):
    model_f = PySRRegressor(**pysr_kwargs)
    model_f.fit(X_train_scaled[tr], y_train[tr])
    pred_va = model_f.predict(X_train_scaled[va])
    cv_rmse_folds.append(root_mean_squared_error(y_train[va], pred_va))

cv_rmse_pysr = np.mean(cv_rmse_folds)
cv_rmse_std_pysr = np.std(cv_rmse_folds)

#Fit final PySR on the full training set
pysr = PySRRegressor(**pysr_kwargs)
pysr.fit(X_train_scaled, y_train)

#PySR predictions (best-selected equation)
y_pred_train_pysr = pysr.predict(X_train_scaled)
y_pred_test_pysr = pysr.predict(X_test_scaled)

#Verify that the selected equation is finite across both the training and test sets
n_nonfinite = int((~np.isfinite(y_pred_train_pysr)).sum() + (~np.isfinite(y_pred_test_pysr)).sum())
if n_nonfinite:
    print(f"Warning: {n_nonfinite} non-finite predictions")

#PySR performance metrics
train_r2_pysr = r2_score(y_train, y_pred_train_pysr)
test_r2_pysr = r2_score(y_test, y_pred_test_pysr)
train_rmse_pysr = root_mean_squared_error(y_train, y_pred_train_pysr)
test_rmse_pysr = root_mean_squared_error(y_test, y_pred_test_pysr)
train_mae_pysr = mean_absolute_error(y_train, y_pred_train_pysr)
test_mae_pysr = mean_absolute_error(y_test, y_pred_test_pysr)

#Store PySR selected equation and tuned parameters
best_eq = pysr.get_best()
pysr_params = {
    'complexity': int(best_eq['complexity']),
    'equation': str(best_eq['equation']),
    'maxsize': pysr_maxsize,
    'parsimony': pysr_parsimony,
    'niterations': pysr_base['niterations'],
    'binary_operators': pysr_base['binary_operators'],
    'unary_operators': pysr_base['unary_operators'],
    'model_selection': pysr_base['model_selection'],
}

#Store PySR results
results.append({
    'model': 'PySR',
    'cv_rmse': round(cv_rmse_pysr, 4),
    'cv_rmse_std': round(cv_rmse_std_pysr, 4),
    'train_r2': round(train_r2_pysr, 4),
    'test_r2': round(test_r2_pysr, 4),
    'train_rmse': round(train_rmse_pysr, 4),
    'test_rmse': round(test_rmse_pysr, 4),
    'train_mae': round(train_mae_pysr, 4),
    'test_mae': round(test_mae_pysr, 4),
    'best_params': str(pysr_params)
})

#Store PySR predictions
all_predictions = pd.concat([all_predictions, pd.DataFrame({
    'metal_group': groups_train,
    'model': 'PySR',
    'split': 'train',
    'target_actual': y_train,
    'target_predicted': y_pred_train_pysr
}), pd.DataFrame({
    'metal_group': groups_test,
    'model': 'PySR',
    'split': 'test',
    'target_actual': y_test,
    'target_predicted': y_pred_test_pysr
})], ignore_index=True)

#Record all PySR hyperparameter configurations for Rashomon set
cv_df_pysr_out = cv_df_pysr.copy()
cv_df_pysr_out.insert(0, 'model', 'PySR')
hyperparam_configs = pd.concat([hyperparam_configs, cv_df_pysr_out], ignore_index=True)

#Export all equations with the best one flagged
eq_export = pysr.equations_.copy()
eq_export['selected'] = eq_export['complexity'] == best_eq['complexity']
eq_export.to_excel(os.path.join(desktop_output, f'pysr_{target_col}_all_equations.xlsx'), index=False)

#Export results of all models (single split) and the hyperparameter configurations
pd.DataFrame(results).to_excel(os.path.join(desktop_output, f'ml_results_{target_col}.xlsx'), index=False)
all_predictions.to_excel(os.path.join(desktop_output, f'ml_predictions_{target_col}.xlsx'), index=False)
shap_results.to_excel(os.path.join(desktop_output, f'shap_results_{target_col}.xlsx'), index=False)
hyperparam_configs.to_excel(os.path.join(desktop_output, f'hyperparam_configs_{target_col}.xlsx'), index=False)

## 5.5. Constructing a Rashomon Set

In [ ]:
#Rashomon set construction
cfg = hyperparam_configs.copy()

if 'cv_rmse_sd' in cfg.columns:
    cfg['cv_rmse_std'] = cfg['cv_rmse_std'].fillna(cfg['cv_rmse_sd'])

#Classify every sampled configuration
scale = y_train.std()
cfg['status'] = np.where(cfg['cv_rmse'].isna(), 'failed',
                  np.where(cfg['cv_rmse'] > 10 * scale, 'diverged', 'usable'))
usable = cfg[cfg['status'] == 'usable'].copy()

#Rashomon set threshold, anchored to the fold-to-fold cross-validation RMSE variability of the best model
    #RASHOMON_SD = number of fold-to-fold standard deviations above the best configuration
    #Increase number of standard deviations to enlarge the Rashomon set, then re-run the construction and refitting
RASHOMON_SD = 1
best = usable.loc[usable['cv_rmse'].idxmin()]
eps = best['cv_rmse_std']
if not np.isfinite(eps):
    eps = usable['cv_rmse_std'].min()
    print("Warning: No fold-to-fold standard deviation for the best model")
rashomon_threshold_fixed = best['cv_rmse'] + RASHOMON_SD * eps
usable['member'] = usable['cv_rmse'] <= rashomon_threshold_fixed

#Candidate pool and Rashomon set members by architecture
ratio_rows = []
for m, g in cfg.groupby('model'):
    u = usable[usable.model == m]
    ratio_rows.append({'reaction': reaction, 'architecture': m,
                       'sampled': len(g), 'usable': len(u),
                       'failed': int((g.status == 'failed').sum()),
                       'diverged': int((g.status == 'diverged').sum()),
                       'members': int(u.member.sum()), 'ratio': u.member.mean(),
                       'best_cv_rmse': u.cv_rmse.min(),
                       'median_cv_rmse': u.cv_rmse.median(),
                       'threshold': rashomon_threshold_fixed, 'units': units})

members_fixed = sorted(usable.loc[usable.member, 'model'].unique(),
                       key=lambda m: usable.loc[usable.model == m, 'cv_rmse'].min())
rashomon_ratio = pd.DataFrame(ratio_rows)
candidate_pool = usable

#Export results
rashomon_ratio.round(4).to_excel(
    os.path.join(desktop_output, f'rashomon_ratio_{target_col}.xlsx'), index=False)
candidate_pool.round(6).to_excel(
    os.path.join(desktop_output, f'candidate_model_pool_{target_col}.xlsx'), index=False)

## 5.6. Refitting the Rashomon Set

In [ ]:
#Refitting the Rashomon set across configurations and training seeds
    #Every member of the Rashomon set is refitted on the same training rows and evaluated on the same test set
    #Models differ from one another only in architecture, hyperparameter configuration, and the random seed used during training
    #Each fitted model is a competing account of the same dataset and three nested sources of variation are represented
    #Models sharing a configuration differ only by training seed; models sharing an architecture differ by configuration; 
    #models of different architectures differ by algorithm

#Number of configurations per architecture (above actual maximum to clear the maximum), and training seeds per configuration
N_CONFIGS_PER_ARCH = 2000
TRAIN_SEEDS = [0, 1, 2]

#Configurations to fit in this run (batched)
BATCH_SIZE = None

#Order in which configurations are fitted
SHUFFLE_SEED = 42

REFIT_PATH = os.path.join(desktop_output, f'rashomon_refit_dac_{target_col}.csv')
SHAP_PATH_FIXED = os.path.join(desktop_output, f'rashomon_refit_shap_{target_col}.csv')

#Estimator constructors
_ESTIMATORS = {
    'RFR': (RandomForestRegressor, {}),
    'GBR': (GradientBoostingRegressor, {}),
    'SVR': (SVR, {}),
    'GPR': (GaussianProcessRegressor, {'normalize_y': True}),
    'MLP': (MLPRegressor, {'max_iter': 2000, 'early_stopping': False}),
    'LR':  (LinearRegression, {}),
}

#Deterministic architectures whose fit does not depend on a random seed
_DETERMINISTIC = {'SVR', 'LR'}

#Kernel architecture
_KERNEL_LOOKUP = {str(k): k for k in kernels_gpr}

def build_estimator(row, seed):
    """Reconstruct one sampled configuration"""
    arch = row['model']
    cls, fixed = _ESTIMATORS[arch]
    params = {c: row[c] for c in candidate_pool.attrs['param_cols']
              if c in row.index and pd.notna(row[c])}
    for p in ('n_estimators', 'max_depth', 'min_samples_split', 'min_samples_leaf',
              'max_leaf_nodes', 'degree', 'n_restarts_optimizer'):
        if p in params:
            params[p] = int(params[p])
    if arch == 'GPR':
        params['kernel'] = _KERNEL_LOOKUP[params['kernel']]
    if 'random_state' in cls().get_params():
        params['random_state'] = seed
    return cls(**{**fixed, **params})

#Columns of the candidate pool that describe a configuration
candidate_pool.attrs['param_cols'] = [
    c for c in candidate_pool.columns
    if c not in ('model', 'status', 'member', 'cv_rmse', 'cv_rmse_std', 'cv_rmse_sd',
                 'mean_complexity', 'minutes')]

#Select configurations to refit
selected = []
for arch, g in candidate_pool[candidate_pool.member].groupby('model'):
    if arch not in _ESTIMATORS:
        print(f"  {arch} skipped - not reconstructable")
        continue
    g = g.sort_values('cv_rmse')
    take = g if len(g) <= N_CONFIGS_PER_ARCH else g.iloc[
        np.linspace(0, len(g) - 1, N_CONFIGS_PER_ARCH).astype(int)]
    take = take.copy()
    take['config_id'] = [f"{arch}_{i:03d}" for i in range(len(take))]
    selected.append(take)
selected = pd.concat(selected, ignore_index=True)
selected = selected.sample(frac=1, random_state=SHUFFLE_SEED).reset_index(drop=True)

#Resume from whichever configurations are already recorded
def _load(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        return df, set(df['config_id'].unique())
    return pd.DataFrame(), set()

rank_prev, rank_done = _load(REFIT_PATH)
shap_prev, shap_done = _load(SHAP_PATH_FIXED)
done_configs = rank_done & shap_done
rank_records = (rank_prev[rank_prev['config_id'].isin(done_configs)].to_dict('records')
                if len(rank_prev) else [])
shap_records = (shap_prev[shap_prev['config_id'].isin(done_configs)].to_dict('records')
                if len(shap_prev) else [])

#Configurations in this run that still need fitting
batch_todo = selected[~selected['config_id'].isin(done_configs)]
if BATCH_SIZE is not None:
    batch_todo = batch_todo.head(BATCH_SIZE)

#Fixed reference set
ref_groups = test_df['metal_group'].values

#Save with retries
failures = []
def _save(retries=3):
    for attempt in range(retries):
        try:
            pd.DataFrame(rank_records).to_csv(REFIT_PATH, index=False)
            pd.DataFrame(shap_records).to_csv(SHAP_PATH_FIXED, index=False)
            return True
        except OSError as err:
            print(f"Save failed ({type(err).__name__}); retry {attempt + 1}/{retries}")
            time.sleep(5)
    print("Warning: Save failed after retries")
    return False

t_start = time.time()
for n, (_, row) in enumerate(batch_todo.iterrows(), 1):
    arch = row['model']
    seeds = [0] if arch in _DETERMINISTIC else TRAIN_SEEDS
    for s in seeds:
        try:
            model = build_estimator(row, s)
            model.fit(X_train_scaled, y_train)
            pred = model.predict(X_test_scaled)

            #DAC rankings
            order = np.argsort(np.abs(pred) if target_col == 'delta_G_H' else pred)
            for r, idx in enumerate(order):
                rank_records.append({'architecture': arch, 'config_id': row['config_id'],
                                     'train_seed': s, 'metal_group': ref_groups[idx],
                                     'pred': pred[idx], 'rank': r + 1})

            #SHAP rankings
            sv = shap.KernelExplainer(model.predict, shap_background).shap_values(
                X_test_scaled, nsamples=shap_nsamples, l1_reg=shap_l1_reg)
            for d, val in zip(desc_cols_pruned, np.abs(sv).mean(axis=0)):
                shap_records.append({'architecture': arch, 'config_id': row['config_id'],
                                     'train_seed': s, 'descriptor': d,
                                     'mean_abs_shap': val})
        except Exception as e:
            failures.append({'architecture': arch, 'config_id': row['config_id'],
                             'train_seed': s, 'error': type(e).__name__})

    #Incremental save
    if n % 20 == 0 or n == len(batch_todo):
        _save()
        el = time.time() - t_start
        print(f"  {n}/{len(batch_todo)} this run | "
              f"{len(done_configs) + n}/{len(selected)} overall | "
              f"{el/60:.1f} min elapsed | ~{(len(batch_todo) - n) * el / n / 60:.1f} min left")

_save()
refit_ranks = pd.DataFrame(rank_records)
refit_shap = pd.DataFrame(shap_records)

#Print completion and any failed fits
print(f"{refit_ranks['config_id'].nunique()}/{len(selected)} configurations complete; {len(failures)} failures this run")
if failures:
    print(pd.DataFrame(failures).groupby(['architecture', 'error']).size().to_string())

## 5.7. Consensus Across the Rashomon Set

### 5.7.1. Agreement Within a Configuration, Within an Architecture, and Across Architectures

In [ ]:
#Consensus across the Rashomon set
    #Agreement measured within a configuration, within an architecture, and across architectures

REFIT_PATH = os.path.join(desktop_output, f'rashomon_refit_dac_{target_col}.csv')
SHAP_PATH_FIXED = os.path.join(desktop_output, f'rashomon_refit_shap_{target_col}.csv')

def consensus_levels(df, item_col, val_col):
    """Spearman agreement within a configuration, within an architecture, and across architectures"""
    wide = (df.drop_duplicates(subset=['architecture', 'config_id', 'train_seed', item_col])
              .pivot_table(index=['architecture', 'config_id', 'train_seed'],
                           columns=item_col, values=val_col)
              .sort_index())
    C = np.corrcoef(rankdata(wide.to_numpy(), axis=1))
    arch = wide.index.get_level_values('architecture').to_numpy()
    cfg = wide.index.get_level_values('config_id').to_numpy()

    iu = np.triu_indices(len(C), k=1)
    same_cfg = (cfg[:, None] == cfg[None, :])[iu]
    same_arch = (arch[:, None] == arch[None, :])[iu]
    rho = C[iu]
    ok = np.isfinite(rho)

    def _stat(mask):
        v = rho[mask & ok]
        return (v.mean(), v.std(), v.size) if v.size else (np.nan, np.nan, 0)

    #Agreement between each pair of architectures, and the mean over those pairs
    archs = sorted(set(arch))
    pair_matrix = pd.DataFrame(np.nan, index=archs, columns=archs, dtype=float)
    pair_means = []
    for i in range(len(archs)):
        for j in range(i + 1, len(archs)):
            m = (((arch[:, None] == archs[i]) & (arch[None, :] == archs[j])) |
                 ((arch[:, None] == archs[j]) & (arch[None, :] == archs[i])))[iu]
            v = rho[m & ok]
            if v.size:
                pair_matrix.iloc[i, j] = pair_matrix.iloc[j, i] = v.mean()
                pair_means.append(v.mean())
    #How much an architecture agrees with itself across configurations
    for i, a in enumerate(archs):
        m = ((arch[:, None] == a) & (arch[None, :] == a))[iu] & ~same_cfg
        v = rho[m & ok]
        if v.size:
            pair_matrix.iloc[i, i] = v.mean()

    #Seed and configuration agreement within each architecture, then averaged over architectures
    seed_by_arch, cfg_by_arch = [], []
    for a in archs:
        m = ((arch[:, None] == a) & (arch[None, :] == a))[iu] & ok
        s, c = rho[m & same_cfg], rho[m & ~same_cfg]
        seed_by_arch.append(s.mean() if s.size else 1.0)   
        cfg_by_arch.append(c.mean() if c.size else np.nan)

    return {'seed': (np.mean(seed_by_arch), np.std(seed_by_arch), len(seed_by_arch)),
            'config': (np.nanmean(cfg_by_arch), np.nanstd(cfg_by_arch), int(np.isfinite(cfg_by_arch).sum())),
            'arch': (np.mean(pair_means), np.std(pair_means), len(pair_means)),
            'arch_unweighted': _stat(~same_arch),
            'pair_matrix': pair_matrix,
            'wide': wide, 'C': C, 'arch_labels': arch, 'cfg_labels': cfg}

results_levels = {}
for axis, path, item_col, val_col in [
        ('DAC Ranking', REFIT_PATH, 'metal_group', 'rank'),
        ('SHAP Attribution', SHAP_PATH_FIXED, 'descriptor', 'mean_abs_shap')]:
    results_levels[axis] = consensus_levels(pd.read_csv(path), item_col, val_col)

#Seed and configuration agreement of each architecture
arch_rows = []
for a in sorted(set(results_levels['DAC Ranking']['arch_labels'])):
    vals = {}
    for axis in results_levels:
        r = results_levels[axis]
        m = r['arch_labels'] == a
        sub = r['C'][np.ix_(m, m)]
        c_sub = r['cfg_labels'][m]
        iu = np.triu_indices(len(sub), k=1)
        same = (c_sub[:, None] == c_sub[None, :])[iu]
        v = sub[iu]
        ok = np.isfinite(v)
        vals[axis] = (v[same & ok].mean() if (same & ok).any() else np.nan,
                      v[~same & ok].mean() if (~same & ok).any() else np.nan)
    d_s, d_c = vals['DAC Ranking']
    s_s, s_c = vals['SHAP Attribution']
    arch_rows.append({'reaction': reaction, 'architecture': a,
                      'dac_seed_rho': d_s, 'dac_config_rho': d_c,
                      'shap_seed_rho': s_s, 'shap_config_rho': s_c})

#Values carried forward
consensus_fixed = pd.DataFrame([
    {'reaction': reaction, 'axis': axis,
     'seed_rho': results_levels[axis]['seed'][0],
     'seed_archs': results_levels[axis]['seed'][2],
     'config_rho': results_levels[axis]['config'][0],
     'config_archs': results_levels[axis]['config'][2],
     'arch_rho': results_levels[axis]['arch'][0],
     'arch_pairs': results_levels[axis]['arch'][2],
     'arch_rho_by_instance': results_levels[axis]['arch_unweighted'][0]}
    for axis in results_levels])
consensus_by_arch = pd.DataFrame(arch_rows)
consensus_pairs = {axis: results_levels[axis]['pair_matrix'] for axis in results_levels}

#Contribution of each source, one row per axis
consensus_contributions = pd.DataFrame([
    {'reaction': reaction, 'axis': axis,
     'training_seed': 1.0 - results_levels[axis]['seed'][0],
     'hyperparameters': results_levels[axis]['seed'][0] - results_levels[axis]['config'][0],
     'architecture': results_levels[axis]['config'][0] - results_levels[axis]['arch'][0],
     'total': 1.0 - results_levels[axis]['arch'][0]}
    for axis in results_levels])

#Export results
with pd.ExcelWriter(os.path.join(
        desktop_output, f'rashomon_consensus_{target_col}.xlsx')) as writer:
    consensus_fixed.round(4).to_excel(writer, sheet_name='Levels', index=False)
    consensus_contributions.round(4).to_excel(writer, sheet_name='Contributions', index=False)
    consensus_by_arch.round(4).to_excel(writer, sheet_name='By-Architecture', index=False)
    for axis, pm in consensus_pairs.items():
        pm.round(4).to_excel(writer, sheet_name=f'Pairs-{axis.split()[0]}')

### 5.7.2. Vote-Based Consensus

In [ ]:
#Vote-based consensus across the Rashomon set
    #Each refitted model votes for its top five DACs and its top five descriptors
    #A DAC or descriptor reaches a majority consensus if its vote share is greater than 50%
    #Proportional weighting counts every refitted model once, so an architecture's influence scales with its number of refits
    #Equal weighting averages the vote share within each architecture, such that every architecture counts once

#Define top N and majority voting criteria
TOP_N = 5 
MAJORITY = 0.50
KEY = ['architecture', 'config_id', 'train_seed']

#Top-ranked DACs and descriptors of every refitted model
top_dac = refit_ranks[refit_ranks['rank'] <= TOP_N]
top_desc = (refit_shap.sort_values('mean_abs_shap', ascending=False)
            .groupby(KEY).head(TOP_N))

#Number of refitted models per architecture
n_models = top_dac[KEY].drop_duplicates().groupby('architecture').size()

#Vote share of every DAC and descriptor under both weightings
votes = {}
for kind, df, col in [('DAC', top_dac, 'metal_group'), ('Descriptor', top_desc, 'descriptor')]:
    per_arch = df.groupby(['architecture', col]).size().unstack(fill_value=0).div(n_models, axis=0)
    table = pd.DataFrame({'proportional': df.groupby(col).size() / n_models.sum(),
                          'equal': per_arch.mean(axis=0)})
    table = table.sort_values('proportional', ascending=False)
    table['majority_proportional'] = table['proportional'] > MAJORITY
    table['majority_equal'] = table['equal'] > MAJORITY
    votes[kind] = table.rename_axis(col)

#Export results
with pd.ExcelWriter(os.path.join(desktop_output, f'rashomon_votes_{target_col}.xlsx')) as writer:
    votes['DAC'].round(4).to_excel(writer, sheet_name='DAC')
    votes['Descriptor'].round(4).to_excel(writer, sheet_name='Descriptor')

### 5.7.3. Agreement on Direction of SHapley Additive exPlanation Effects

In [ ]:
#SHAP direction of each descriptor
    #Spearman correlation between a descriptor's raw value and its signed SHAP value across the test set

#Signed SHAP values of the fixed-split models
signed_shap = {'LR': shap_values_lr, 'RFR': shap_values_rfr, 'GBR': shap_values_gbr,
               'SVR': shap_values_svr, 'GPR': shap_values_gpr, 'MLP': shap_values_mlp}

direction_df = pd.DataFrame([
    {'architecture': arch, 'descriptor': d, 'rho': spearmanr(X_test[:, j], sv[:, j]).correlation}
    for arch, sv in signed_shap.items()
    for j, d in enumerate(desc_cols_pruned)])

#Agreement on SHAP sign 
direction_wide = direction_df.pivot(index='descriptor', columns='architecture', values='rho')[list(signed_shap)]
in_set = direction_wide[[a for a in direction_wide.columns if a in members_fixed]]
direction_wide['unanimous_in_rashomon_set'] = in_set.apply(
    lambda r: bool((np.sign(r) == np.sign(r.iloc[0])).all()), axis=1)

#Signed SHAP values
signed_long = pd.concat([
    pd.DataFrame(sv, columns=desc_cols_pruned).assign(metal_group=test_df['metal_group'].values, architecture=arch)
    for arch, sv in signed_shap.items() if arch in members_fixed
]).melt(id_vars=['metal_group', 'architecture'], var_name='descriptor', value_name='signed_shap')

#Export results
direction_wide.round(4).to_excel(os.path.join(desktop_output, f'shap_direction_{target_col}.xlsx'))
signed_long.round(6).to_excel(os.path.join(desktop_output, f'shap_signed_{target_col}.xlsx'), index=False)

## 5.8. Model Performance Under Resampling

### 5.8.1. MP, LR, RFR, GBR, SVR, GPR, MLP

In [ ]:
#Resampling across 1000 random seeds
seeds = list(range(1000))

#Define X, y, groups from final descriptor set
X = dataset_descriptors_pruned[desc_cols_pruned].values
y = dataset_descriptors_pruned[target_col].values
groups = dataset_descriptors_pruned['metal_group'].values

#Define models
robustness_models = {
    'MP': DummyRegressor(strategy='mean'),
    'LR': LinearRegression(),
    'RFR': clone(rfr),
    'GBR': clone(gbr),
    'SVR': clone(svr),
    'GPR': clone(gpr),
    'MLP': clone(mlp),
}

robustness_models['RFR'].set_params(n_jobs=-1)
    
#Storage of results
robustness_results = []

t_start = time.time()

for seed in seeds:
    #Re-split with seed, test_size = 0.20 
    gss_rob = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, test_idx = next(gss_rob.split(X, y, groups=groups))
    X_train_rob, X_test_rob = X[train_idx], X[test_idx]
    y_train_rob, y_test_rob = y[train_idx], y[test_idx]

    #Scale per seed and fit on train only 
    scaler_rob = StandardScaler()
    X_train_rob_scaled = scaler_rob.fit_transform(X_train_rob)
    X_test_rob_scaled = scaler_rob.transform(X_test_rob)

    for name, model in robustness_models.items():
        try:
            m = clone(model)
            m.fit(X_train_rob_scaled, y_train_rob)
            y_pred_test_rob = m.predict(X_test_rob_scaled)
            y_pred_train_rob = m.predict(X_train_rob_scaled)
            robustness_results.append({
                'seed': seed,
                'model': name,
                'train_rmse': round(root_mean_squared_error(y_train_rob, y_pred_train_rob), 4),
                'test_rmse': round(root_mean_squared_error(y_test_rob, y_pred_test_rob), 4),
                'train_r2': round(r2_score(y_train_rob, y_pred_train_rob), 4),
                'test_r2': round(r2_score(y_test_rob, y_pred_test_rob), 4),
                'train_mae': round(mean_absolute_error(y_train_rob, y_pred_train_rob), 4),
                'test_mae': round(mean_absolute_error(y_test_rob, y_pred_test_rob), 4),
            })
        except Exception as e:
            print(f"Seed {seed} {name}: Failed — {e}")
    
    if (seed + 1) % 10 == 0:
        el = time.time() - t_start
        print(f"Completed {seed + 1}/{len(seeds)} seeds | {el/60:.1f} min elapsed | "
              f"~{(len(seeds) - seed - 1) * el / (seed + 1) / 60:.1f} min left")
    #Incremental save
    if (seed + 1) % 50 == 0:
        pd.DataFrame(robustness_results).to_excel(
            os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'), index=False)

#Export results
pd.DataFrame(robustness_results).to_excel(
    os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'), index=False)

### 5.8.2. PySR

In [ ]:
#Identical PySR configuration except parallel and non-deterministic
pysr_rob_kwargs = {**pysr_base, **pysr_parallel,
                   'maxsize': pysr_maxsize, 'parsimony': pysr_parsimony}

#Seeds to run 
PYSR_SEEDS = list(range(0, 1000))
TOTAL_SEEDS = 1000

PYSR_ROB_PATH = os.path.join(desktop_output, f'seed_resampling_pysr_{target_col}.xlsx')

#Load descriptors
X = dataset_descriptors_pruned[desc_cols_pruned].values
y = dataset_descriptors_pruned[target_col].values
groups = dataset_descriptors_pruned['metal_group'].values

#Resume from any seeds already recorded
if os.path.exists(PYSR_ROB_PATH):
    pysr_rob_df = pd.read_excel(PYSR_ROB_PATH)
    done_seeds = set(pysr_rob_df['seed'].tolist())
    pysr_rob_results = pysr_rob_df.to_dict('records')
else:
    done_seeds = set()
    pysr_rob_results = []

#Seeds in this batch that still need running
batch_todo = [s for s in PYSR_SEEDS if s not in done_seeds]

t_start = time.time()
for seed in PYSR_SEEDS:
    if seed in done_seeds:
        continue

    #Re-split at this seed
    gss_rob = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    tr_idx, te_idx = next(gss_rob.split(X, y, groups=groups))
    X_tr, X_te = X[tr_idx], X[te_idx]
    y_tr, y_te = y[tr_idx], y[te_idx]

    #Scale per seed and fit on train only 
    scaler_rob = StandardScaler()
    X_tr_s = scaler_rob.fit_transform(X_tr)
    X_te_s = scaler_rob.transform(X_te)

    #One PySR fit per seed
    try:
        m = PySRRegressor(**pysr_rob_kwargs)
        m.fit(X_tr_s, y_tr)
        yp_te = m.predict(X_te_s)
        yp_tr = m.predict(X_tr_s)
        best_eq_seed = m.get_best()
        pysr_rob_results.append({
            'seed': seed,
            'model': 'PySR',
            'train_rmse': round(root_mean_squared_error(y_tr, yp_tr), 4),
            'test_rmse': round(root_mean_squared_error(y_te, yp_te), 4),
            'train_r2': round(r2_score(y_tr, yp_tr), 4),
            'test_r2': round(r2_score(y_te, yp_te), 4),
            'train_mae': round(mean_absolute_error(y_tr, yp_tr), 4),
            'test_mae': round(mean_absolute_error(y_te, yp_te), 4),
            'complexity': int(best_eq_seed['complexity']),
            'loss': round(float(best_eq_seed['loss']), 6),
            'equation': str(best_eq_seed['equation']),
        })
    except Exception as e:
        print(f"Seed {seed}: FAILED — {e}")
        continue

    #Progress and incremental save
    n_done = len(pysr_rob_results) - len(done_seeds)
    if n_done % 10 == 0 and n_done > 0:
        pd.DataFrame(pysr_rob_results).to_excel(PYSR_ROB_PATH, index=False)
        elapsed = time.time() - t_start
        remaining = (len(batch_todo) - n_done) * elapsed / n_done
        print(f"  {len(done_seeds) + n_done}/{TOTAL_SEEDS} seeds | "
              f"{elapsed/3600:.2f}h elapsed | ~{remaining/3600:.2f}h left")

#Save seeds
pd.DataFrame(pysr_rob_results).to_excel(PYSR_ROB_PATH, index=False)

## 5.9. Model Comparison Under Resampling

In [ ]:
#Comparison of performance across all models when the train/test partition itself is varied

#Load seed resampling results and combine
rob_df = pd.read_excel(os.path.join(desktop_output, f'seed_resampling_{target_col}.xlsx'))
pysr_df = pd.read_excel(os.path.join(desktop_output, f'seed_resampling_pysr_{target_col}.xlsx'))
rob_df = pd.concat([rob_df, pysr_df], ignore_index=True)

#Define performance of each model across seed resampling
resampling_stats = rob_df.groupby('model').agg(
    train_r2_mean=('train_r2', 'mean'), train_r2_std=('train_r2', 'std'),
    test_r2_mean=('test_r2', 'mean'), test_r2_std=('test_r2', 'std'),
    train_rmse_mean=('train_rmse', 'mean'), train_rmse_std=('train_rmse', 'std'),
    test_rmse_mean=('test_rmse', 'mean'), test_rmse_std=('test_rmse', 'std'),
    train_mae_mean=('train_mae', 'mean'), train_mae_std=('train_mae', 'std'),
    test_mae_mean=('test_mae', 'mean'), test_mae_std=('test_mae', 'std'),
    n_seeds=('test_rmse', 'count'),
).reset_index()

#Order by performance
resampling_stats = resampling_stats.sort_values('test_rmse_mean').reset_index(drop=True)

#Identify win-rate — the fraction of seeds each model has the lowest test RMSE
n_seeds_total = rob_df['seed'].nunique()
win_counts = rob_df.loc[rob_df.groupby('seed')['test_rmse'].idxmin(), 'model'].value_counts()
resampling_stats['win_rate'] = (resampling_stats['model'].map(win_counts).fillna(0)
                                / n_seeds_total * 100).round(1)

#Format per-model results
def ms(m, s, dp=3):
    return f"{m:.{dp}f} ± {s:.{dp}f}"
summary_table = pd.DataFrame({
    'Model': resampling_stats['model'],
    'Train R²': [ms(m, s) for m, s in zip(resampling_stats['train_r2_mean'],
                                          resampling_stats['train_r2_std'])],
    'Test R²': [ms(m, s) for m, s in zip(resampling_stats['test_r2_mean'],
                                         resampling_stats['test_r2_std'])],
    'Train RMSE': [ms(m, s) for m, s in zip(resampling_stats['train_rmse_mean'],
                                            resampling_stats['train_rmse_std'])],
    'Test RMSE': [ms(m, s) for m, s in zip(resampling_stats['test_rmse_mean'],
                                           resampling_stats['test_rmse_std'])],
    'Train MAE': [ms(m, s) for m, s in zip(resampling_stats['train_mae_mean'],
                                           resampling_stats['train_mae_std'])],
    'Test MAE': [ms(m, s) for m, s in zip(resampling_stats['test_mae_mean'],
                                          resampling_stats['test_mae_std'])],
    'Win Rate': [f"{w:.1f}%" for w in resampling_stats['win_rate']],
})

#Export results
with pd.ExcelWriter(os.path.join(
        desktop_output, f'model_comparison_resampling_{target_col}.xlsx')) as writer:
    summary_table.to_excel(writer, sheet_name='Formatted', index=False)
    resampling_stats.round(4).to_excel(writer, sheet_name='Numeric', index=False)